# Gauss's principle of least constraint enables neural solution of forward and inverse dynamics


A key challenge in sensorimotor neuroscience is figuring out how the brain controls/adapts movement in real time, since **_sensory feedback is often too slow_** for fast actions. Most believe the central nervous system (CNS) uses **_internal models_**, or neural representations of the body’s physics, to predict and plan movements.

Traditionally, these models are separated into two mathematical types:

* **_Forward models_** predict the body’s future state, such as position and velocity, based on current motor commands. They are needed for **_anticipating movement_** and estimating the body’s state.
* **_Inverse models_** do the opposite: they calculate the motor commands needed to reach a desired future state, which is needed for **_planning movement paths_**.

Mainstream belief is that these two functions need **_separate neural circuits_**. Here, we ask whether this separation is truly biologically necessary, or whether it results from the **_mathematical models_** often used, such as Newton-Euler mechanics, which treat forward and inverse dynamics as **_distinct problems_**.

We propose an alternative approach using the **_Gibbs-Appell formulation of motion_**. We demonstrate that by using this formulation, derived from **_Gauss's principle of least constraint_**, a **_single neural network_** can learn the underlying physics of the body and be used to perform **_both forward and inverse dynamics_** without needing to be trained on them separately.

As such, this work shows that forward/inverse separation might just be a **_mathematical artifact_**, **_not a biological necessity_**! 🎤⬇💥

PS: While we do not claim that the brain avoids functional separation, our results show that a **_single neural substrate_** could theoretically support **_both directions of dynamic transformation_**, which **_challenges the idea_** that forward and inverse models must be separate computational entities.


In [2]:
# Install all third-party libraries used in this notebook for local Jupyter execution.
# google.colab is handled later as an optional Colab-only helper for browser downloads.
import importlib.util
import subprocess
import sys

PIP_DEPENDENCIES = {
    'psutil': 'psutil',
    'numpy': 'numpy',
    'sympy': 'sympy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'statsmodels': 'statsmodels',
    'torch': 'torch',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
}

missing_packages = [
    package_name
    for module_name, package_name in PIP_DEPENDENCIES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"Installing missing pip packages: {', '.join(missing_packages)}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
    print('If any packages were installed, rerun this cell or restart the kernel and run the notebook from the top.')
else:
    print('All third-party notebook dependencies are already installed.')


All third-party notebook dependencies are already installed.


### Libraries import

In [3]:
# System & environment utilities
import os
import sys
import time
import psutil
import subprocess
from itertools import combinations

# Core numerics, statistics, and symbolic computation
import numpy as np
import sympy as sp
import pandas as pd
import scipy.stats as scipy_stats
from scipy.optimize import minimize
from statsmodels.stats.multitest import multipletests

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
from torch.utils.data import Dataset, DataLoader, Subset

# Visualization
import seaborn as sns
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors


### Style setup

The code below sets Matplotlib parameters to match Nature Communications figure style, including fonts, line widths, colors, and recommended figure sizes.

In [4]:
def set_nature_style():
    """
    Configure matplotlib for Nature Communications style:
    - Sans-serif font (Helvetica/Arial preferred)
    - Font size 5–7 pt for axes/legends, 7 pt for panel labels
    - Figure widths: 89 mm (single) or 183 mm (double column)
    - Line widths: 0.5–1 pt
    - Use high-res (600 dpi) for raster exports
    """

    # 1. Figure widths in inches (Nature standard)
    NATURE_SINGLE_COL_WIDTH = 3.50  # 89 mm
    NATURE_DOUBLE_COL_WIDTH = 7.20  # 183 mm

    # Golden ratio for height if not specified
    GOLDEN_RATIO = (5**0.5 - 1) / 2

    # 2. Update rcParams for fonts, lines, ticks, saving, and aesthetics
    nature_rc = {
        # Fonts
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 7,            # Base font size
        'axes.labelsize': 7,       # Axis labels
        'axes.titlesize': 7,       # Subplot titles
        'xtick.labelsize': 6,      # Tick labels
        'ytick.labelsize': 6,
        'legend.fontsize': 6,      # Legend text

        # === FIX FOR ADOBE ILLUSTRATOR TEXT EDITING ===
        'pdf.fonttype': 42,        # Output Type 42 (TrueType) fonts for PDF
        'ps.fonttype': 42,         # Output Type 42 (TrueType) fonts for PostScript
        'svg.fonttype': 'none',    # Export text as text elements, not paths for SVG
        # ==============================================

        # Lines & ticks
        'axes.linewidth': 0.5,     # Axes edge lines
        'grid.linewidth': 0.5,
        'lines.linewidth': 1.0,    # Plot lines
        'lines.markersize': 3,
        'xtick.major.width': 0.5,
        'ytick.major.width': 0.5,
        'xtick.minor.width': 0.4,
        'ytick.minor.width': 0.4,

        # Saving
        'savefig.dpi': 600,        # High-res
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.05,

        # Aesthetics
        'axes.prop_cycle': plt.cycler(
            color=['#E64B35', '#4DBBD5', '#00A087', '#3C5488',
                   '#F39B7F', '#8491B4', '#91D1C2', '#DC0000', '#7E6148']
        ),  # Nature-ish palette
    }

    mpl.rcParams.update(nature_rc)
    return NATURE_SINGLE_COL_WIDTH, NATURE_DOUBLE_COL_WIDTH, GOLDEN_RATIO

# Apply the settings globally
single_col, double_col, golden_ratio = set_nature_style()

print("Nature Communications style applied.")
print("Text will now remain editable in Adobe Illustrator (TrueType 42 / SVG text).")

Nature Communications style applied.
Text will now remain editable in Adobe Illustrator (TrueType 42 / SVG text).


# Section 1: Derivations

In this section, we derive the kinematics of the double-pendulum system and compute the corresponding Gibbs-Appell function, S. Deriving multi-body dynamics by hand is error-prone, so we use SymPy.

In [5]:
# Initialize pretty printing for clearer mathematical output
sp.init_printing(use_unicode=True)

First, we need to define the symbolic variables and functions that describe the system and its motion over time.

In [6]:
# Define time
t = sp.symbols('t')

# Define system parameters
m1, m2 = sp.symbols('m1 m2')
l1, l2 = sp.symbols('l1 l2')
g      = sp.symbols('g')

# Define generalized coordinates as functions of time
theta1 = sp.Function('theta1')(t)
theta2 = sp.Function('theta2')(t)

# Define substitution symbols for positions, velocities, and accelerations
q1, q2     = sp.symbols('q1 q2')
dq1, dq2   = sp.symbols('dq1 dq2')
ddq1, ddq2 = sp.symbols('ddq1 ddq2')

print("Symbols and coordinates defined.")


# Compute position of mass 1
x1 = l1 * sp.sin(theta1)
y1 = -l1 * sp.cos(theta1)

# Compute position of mass 2 relative to link 1
x2 = x1 + l2 * sp.sin(theta1 + theta2)
y2 = y1 - l2 * sp.cos(theta1 + theta2)

print("Position vectors defined.")


# Compute velocities as first derivatives of positions
vx1 = sp.diff(x1, t)
vy1 = sp.diff(y1, t)
vx2 = sp.diff(x2, t)
vy2 = sp.diff(y2, t)

# Compute accelerations as second derivatives of positions
ax1 = sp.diff(vx1, t)
ay1 = sp.diff(vy1, t)
ax2 = sp.diff(vx2, t)
ay2 = sp.diff(vy2, t)

print("Velocity and acceleration vectors defined.")


# Compute squared magnitudes of accelerations
accel_mag_sq_1 = ax1**2 + ay1**2
accel_mag_sq_2 = ax2**2 + ay2**2

# Compute Gibbs-Appell function S = 0.5 * sum(m_i * a_i^2)
S_symbolic = 0.5 * m1 * accel_mag_sq_1 + 0.5 * m2 * accel_mag_sq_2

print("Gibbs-Appell function derived.")


Symbols and coordinates defined.
Position vectors defined.
Velocity and acceleration vectors defined.
Gibbs-Appell function derived.


Second, we replace time-dependent derivatives and generalized coordinates with static symbols to make the final expressions easier to read and work with.

In [7]:
# Create a mapping from dynamic functions to static symbols
# Derivatives must be substituted before the base functions
subs_map = {
    theta1.diff(t, t): ddq1,  # acceleration of joint1
    theta2.diff(t, t): ddq2,  # acceleration of joint2
    theta1.diff(t):    dq1,   # velocity of joint1
    theta2.diff(t):    dq2,   # velocity of joint2
    theta1:             q1,   # angle of joint1
    theta2:             q2    # angle of joint2
}

# Apply the substitutions and simplify the expression
S_clean = S_symbolic.subs(subs_map).simplify()

# Display the simplified Gibbs-Appell function
print("Gibbs-Appell Function:")
sp.pprint(S_clean)


Gibbs-Appell Function:
                                   ⎛                                           ↪
      2    ⎛    2      4⎞          ⎜⎛                     2                    ↪
0.5⋅l₁ ⋅m₁⋅⎝ddq₁  + dq₁ ⎠ + 0.5⋅m₂⋅⎝⎝ddq₁⋅l₁⋅sin(q₁) + dq₁ ⋅l₁⋅cos(q₁) + l₂⋅(d ↪

↪                                                         2                    ↪
↪                                          2             ⎞    ⎛                ↪
↪ dq₁ + ddq₂)⋅sin(q₁ + q₂) + l₂⋅(dq₁ + dq₂) ⋅cos(q₁ + q₂)⎠  + ⎝ddq₁⋅l₁⋅cos(q₁) ↪

↪                                                                              ↪
↪       2                                                            2         ↪
↪  - dq₁ ⋅l₁⋅sin(q₁) + l₂⋅(ddq₁ + ddq₂)⋅cos(q₁ + q₂) - l₂⋅(dq₁ + dq₂) ⋅sin(q₁  ↪

↪       2⎞
↪      ⎞ ⎟
↪ + q₂)⎠ ⎠


Third, we convert the symbolic Gibbs-Appell function into a PyTorch-compatible function so we can evaluate it numerically.

In [8]:
# Convert the symbolic Gibbs-Appell function to a Torch-compatible function
compute_S_ground_truth = sp.lambdify(
    (
        q1, q2,     # positions
        dq1, dq2,   # velocities
        ddq1, ddq2, # accelerations
        m1, m2,     # masses
        l1, l2      # link lengths
    ),
    S_clean,
    "torch",
)

# Perform a sanity check with dummy values
test_val = compute_S_ground_truth(
    torch.tensor(0.0), torch.tensor(0.0),   # q
    torch.tensor(0.0), torch.tensor(0.0),   # dq
    torch.tensor(1.0), torch.tensor(1.0),   # ddq
    torch.tensor(1.0), torch.tensor(1.0),   # m
    torch.tensor(1.0), torch.tensor(1.0),   # l
)

# Print the test value
print(f"Test S value: {test_val}")


Test S value: 5.0


Next, we derive expressions for the generalized forces acting on the system by taking the partial derivatives of the Gibbs-Appell function with respect to the accelerations.

In [9]:
# Compute generalized forces as derivatives of the Gibbs-Appell function
tau1_sym = sp.diff(S_clean, ddq1)  # force corresponding to q1
tau2_sym = sp.diff(S_clean, ddq2)  # force corresponding to q2

# Display generalized force tau1
print("Generalized force τ1:")
sp.pprint(tau1_sym)

print("\n")

# Display generalized force tau2
print("Generalized force τ2:")
sp.pprint(tau2_sym)


Generalized force τ1:
           2             ⎛                                   ⎛                 ↪
1.0⋅ddq₁⋅l₁ ⋅m₁ + 0.5⋅m₂⋅⎝(2⋅l₁⋅sin(q₁) + 2⋅l₂⋅sin(q₁ + q₂))⋅⎝ddq₁⋅l₁⋅sin(q₁)  ↪

↪      2                                                            2          ↪
↪ + dq₁ ⋅l₁⋅cos(q₁) + l₂⋅(ddq₁ + ddq₂)⋅sin(q₁ + q₂) + l₂⋅(dq₁ + dq₂) ⋅cos(q₁ + ↪

↪     ⎞                                      ⎛                     2           ↪
↪  q₂)⎠ + (2⋅l₁⋅cos(q₁) + 2⋅l₂⋅cos(q₁ + q₂))⋅⎝ddq₁⋅l₁⋅cos(q₁) - dq₁ ⋅l₁⋅sin(q₁ ↪

↪                                                   2             ⎞⎞
↪ ) + l₂⋅(ddq₁ + ddq₂)⋅cos(q₁ + q₂) - l₂⋅(dq₁ + dq₂) ⋅sin(q₁ + q₂)⎠⎠


Generalized force τ2:
       ⎛     ⎛                     2                                           ↪
0.5⋅m₂⋅⎝2⋅l₂⋅⎝ddq₁⋅l₁⋅sin(q₁) + dq₁ ⋅l₁⋅cos(q₁) + l₂⋅(ddq₁ + ddq₂)⋅sin(q₁ + q₂ ↪

↪                   2             ⎞                     ⎛                      ↪
↪ ) + l₂⋅(dq₁ + dq₂) ⋅cos(q₁ + q₂)⎠⋅sin(q₁ + q₂) + 2⋅l₂⋅⎝ddq₁⋅l₁⋅cos(q₁) - dq₁ ↪

↪

Lastly, we convert the symbolic generalized forces into a PyTorch-compatible function so we can evaluate them numerically with tensors.

In [10]:
# Convert the symbolic generalized forces to a Torch-compatible function
compute_tau_ground_truth = sp.lambdify(
    (
        q1, q2,     # positions
        dq1, dq2,   # velocities
        ddq1, ddq2, # accelerations
        m1, m2,     # masses
        l1, l2      # link lengths
    ),
    (tau1_sym, tau2_sym),  # outputs: tau1 and tau2
    "torch",
)

# Perform a sanity check with dummy values
test_val = compute_tau_ground_truth(
    torch.tensor(0.0), torch.tensor(0.0),   # q
    torch.tensor(0.0), torch.tensor(0.0),   # dq
    torch.tensor(1.0), torch.tensor(1.0),   # ddq
    torch.tensor(1.0), torch.tensor(1.0),   # m
    torch.tensor(1.0), torch.tensor(1.0),   # l
)

# Print the test values for tau
print(f"Test τ values: {test_val}")


Test τ values: (tensor(7.), tensor(3.))


# Section 2: Initial setup


In this section, we handle the initial configuration and data preparation for a deep learning model designed to simulate a double-pendulum system.

### Constants

In [11]:
# Define physical parameters of the double-pendulum system
PARAMS = {
    'm1': 1.0,
    'm2': 1.0,
    'l1': 1.0,
    'l2': 1.0,
}

# Set the training batch size
BATCH_SIZE = 64

# Set the random seed for reproducibility
SEED = 27
torch.manual_seed(SEED)


In [12]:
# Select GPU if available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the selected device
print(f"Running on device: {device}")


Running on device: cpu


Here, we code up a class that generates a synthetic double-pendulum dataset, computes the corresponding Gibbs-Appell function and generalized forces, and adapts everything for PyTorch training.

In [13]:
class MechanicsDataset(Dataset):
    def __init__(self, n_samples):
        """Describes physics, normalizes data on-device."""

        # Generate data
        q   = torch.empty(n_samples, 2, device=device).uniform_(-torch.pi/2, torch.pi/2)
        dq  = torch.empty(n_samples, 2, device=device).uniform_(-5, 5)
        ddq = torch.empty(n_samples, 2, device=device).uniform_(-20, 20)

        # Cast params to tensors for compatibility
        params = [torch.tensor(PARAMS[k], device=device) for k in ['m1', 'm2', 'l1', 'l2']]

        # Compute Tau (Forces) -> Stack results to (N, 2)
        taus = compute_tau_ground_truth(
            q[:,0], q[:,1], dq[:,0], dq[:,1], ddq[:,0], ddq[:,1], *params
        )
        self.tau = torch.stack(taus, dim=1)

        # Compute S (Gibbs-Appell function) -> Reshape to (N, 1)
        self.S = compute_S_ground_truth(
            q[:,0], q[:,1], dq[:,0], dq[:,1], ddq[:,0], ddq[:,1], *params
        ).reshape(-1, 1)

        self.X = torch.cat([q, dq, ddq], dim=1)

        # Calculate stats (preserved for inference)
        self.X_mean = self.X.mean(dim=0)
        self.X_std  = self.X.std(dim=0)
        self.S_mean = self.S.mean()
        self.S_std  = self.S.std()

        # Apply Z-Score Normalization
        self.X = (self.X - self.X_mean) / (self.X_std + 1e-8)
        self.S = (self.S - self.S_mean) / (self.S_std + 1e-8)

        # Chain rule scaling S_std / ddq_std (indices 4,5)
        self.chain_scale = self.S_std / (self.X_std[4:6] + 1e-8)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Returns tuple: (Inputs, Normalized_Metric, Ground_Truth_Forces)
        return self.X[idx], self.S[idx], self.tau[idx]

# Section 3: Physics-informed training

In this section, we define the feedforward network architecture and set up the training pipeline using the functions created earlier.

### Constants

In [14]:
# Training hyperparameters
EPOCHS = 5000
LR     = 0.0011316535454287552

# Network architecture (number of nodes per hidden layer)
NUM_NODE_L1 = 225
NUM_NODE_L2 = 61
NUM_NODE_L3 = 91
NUM_NODE_L4 = 99

# Dataset sizes to experiment with
dataset_sizes = [500, 600, 700, 800]

# Create output directory if it does not exist
output_dir = "."
os.makedirs(output_dir, exist_ok=True)

# File paths for saving the trained model and training history
MODEL_PATH   = "gibbs_appell_net.pth"
HISTORY_PATH = MODEL_PATH + "_history.pt"


In the class below, we define a multi-layer perceptron that maps the six input features (angle x 2, velocity x 2, and acceleration x 2) of the double-pendulum system to a single output representing the Gibbs-Appell function.

In [15]:
# Network architecture
class GibbsAppellNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Standard feedforward MLP with four hidden layers
        self.net = nn.Sequential(
            nn.Linear(6, NUM_NODE_L1),
            nn.GELU(),
            nn.Linear(NUM_NODE_L1, NUM_NODE_L2),
            nn.GELU(),
            nn.Linear(NUM_NODE_L2, NUM_NODE_L3),
            nn.GELU(),
            nn.Linear(NUM_NODE_L3, NUM_NODE_L4),
            nn.GELU(),
            nn.Linear(NUM_NODE_L4, 1),  # Output layer
        )

    def forward(self, x):
        # Forward pass through the network
        return self.net(x)


Since the network above is expected to predict the Gibbs-Appell function, S,  its automatic differentiation with respect to input accelerations is expected to produce estimations of generalized forces.

Hence, below, we code up the loss that compares these predicted generalized forces to the ground truth, letting the network learn from physics directly without needing to compute S explicitly!


In [16]:
# Physics-informed loss function
def compute_force_loss(model, X_batch, tau_real_batch, chain_scale):
    """
    Computes the physics-informed loss using automatic differentiation.

    Args:
        model: The neural network.
        X_batch: Normalized inputs [N, 6] (requires_grad=True).
        tau_real_batch: Ground-truth forces [N, 2] in real units.
        chain_scale: Precomputed scaling factor for chain-rule correction [2].
    """

    # Predict normalized S from the network
    S_pred_norm = model(X_batch)

    # Compute gradients of S with respect to inputs
    grads_norm = autograd.grad(
        outputs=S_pred_norm.sum(),
        inputs=X_batch,
        create_graph=True
    )[0]

    # Extract acceleration gradients (columns 4 and 5 correspond to ddq1 and ddq2)
    dS_dddq_norm = grads_norm[:, 4:6]

    # Apply chain-rule scaling to convert from normalized to real units
    tau_pred_real = dS_dddq_norm * chain_scale

    # Return mean squared error between predicted and ground-truth forces
    return torch.mean((tau_pred_real - tau_real_batch) ** 2)


Next, to determine how much data the network needs to learn accurate force predictions, we train it on datasets of varying sizes. For each dataset, we generate trajectories, validate periodically, and save the models, statistics, and training history.

In [17]:
# Check if running in Google Colab for browser-based downloads
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab. Files will be saved locally but not downloaded via browser.")

# Loop over dataset sizes to train or skip existing models
for n_samples in dataset_sizes:
    print(f"\n{'='*50}")
    print(f"Processing dataset size: {n_samples}")
    print(f"{'='*50}")

    # Construct unique file paths for model, history, stats, and timing
    model_path = os.path.join(output_dir, f"model_{n_samples}.pth")
    hist_path  = os.path.join(output_dir, f"history_{n_samples}.pt")
    stats_path = os.path.join(output_dir, f"stats_{n_samples}.pt")
    time_path  = os.path.join(output_dir, f"time_{n_samples}.txt")

    print(f"Model path: {model_path}")

    # Skip training if a model already exists for this dataset size
    if os.path.exists(model_path):
        print(f"Found existing model: '{model_path}'")
        print(f"Skipping training for dataset size {n_samples}...")
        continue

    # Generate a fresh dataset
    print(f"Generating dataset with {n_samples} samples...")
    full_dataset = MechanicsDataset(n_samples=n_samples)

    # Save normalization statistics for later inference
    print(f"Saving statistics to '{stats_path}'...")
    norm_stats = {
        'n_samples':   n_samples,
        'X_mean':      full_dataset.X_mean,
        'X_std':       full_dataset.X_std,
        'S_mean':      full_dataset.S_mean,
        'S_std':       full_dataset.S_std,
        'chain_scale': full_dataset.chain_scale
    }
    torch.save(norm_stats, stats_path)
    if IN_COLAB: files.download(stats_path)

    # Split dataset: 80% training, 20% testing
    split_idx     = int(0.8 * len(full_dataset))
    train_dataset = Subset(full_dataset, range(0, split_idx))
    test_dataset  = Subset(full_dataset, range(split_idx, len(full_dataset)))

    # Create DataLoaders for batch processing
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    # Re-initialize the network and optimizer for a fresh start
    model = GibbsAppellNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # Move scaling factors to GPU for efficiency
    chain_scale_gpu = full_dataset.chain_scale.to(device, dtype=torch.float32)

    train_history = []
    test_history  = []
    best_val_loss = float("inf")
    best_epoch    = -1

    print(f"Starting training on {device}...")

    # Synchronize CUDA for accurate timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    # Training loop
    for epoch in range(EPOCHS):
        model.train()           # Enable training mode
        total_loss = 0.0

        for X_batch, _, tau_batch in train_loader:
            X_batch   = X_batch.to(device).requires_grad_(True)  # Needed for physics-informed loss
            tau_batch = tau_batch.to(device)

            optimizer.zero_grad()

            # Compute physics-informed loss
            loss = compute_force_loss(model, X_batch, tau_batch, chain_scale_gpu)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        train_history.append(avg_train_loss)

        # Periodic validation every 100 epochs
        if epoch % 100 == 0:
            model.eval()
            total_val_loss = 0.0

            for X_test, _, tau_test in test_loader:
                X_test   = X_test.to(device).requires_grad_(True)
                tau_test = tau_test.to(device)

                loss_val = compute_force_loss(model, X_test, tau_test, chain_scale_gpu)
                total_val_loss += loss_val.item()

            avg_val_loss = total_val_loss / len(test_loader)
            test_history.append((epoch, avg_val_loss))

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_epoch = epoch
                torch.save(model.state_dict(), model_path)
                print(f"Epoch {epoch:4d} | Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f} | Saved new best model")
            else:
                print(f"Epoch {epoch:4d} | Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f} | Best: {best_val_loss:.6f}")

    # Synchronize CUDA and record elapsed time
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - start_time
    print(f"Training finished in {elapsed_seconds:.2f} seconds.")

    # Save training duration
    with open(time_path, "w") as f:
        f.write(str(elapsed_seconds))
    if IN_COLAB: files.download(time_path)

    if best_epoch >= 0:
        print(f"Best model already saved to '{model_path}' from epoch {best_epoch} with validation loss {best_val_loss:.6f}.")
        if IN_COLAB: files.download(model_path)
    else:
        print("Warning: no validation checkpoint was saved.")

    # Save training history
    print(f"Saving history to '{hist_path}'...")
    torch.save({'train': train_history, 'test': test_history}, hist_path)
    if IN_COLAB: files.download(hist_path)

print("\nAll experiments completed.")


Not running in Google Colab. Files will be saved locally but not downloaded via browser.

Processing dataset size: 500
Model path: ./model_500.pth
Generating dataset with 500 samples...
Saving statistics to './stats_500.pt'...
Starting training on cpu...
Epoch    0 | Train: 1858.931623 | Val: 1682.072510 | Saved new best model
Epoch  100 | Train: 1.390517 | Val: 9.346539 | Saved new best model
Epoch  200 | Train: 0.685198 | Val: 3.812956 | Saved new best model
Epoch  300 | Train: 0.281112 | Val: 3.079980 | Saved new best model
Epoch  400 | Train: 0.390263 | Val: 3.423966 | Best: 3.079980
Epoch  500 | Train: 0.593498 | Val: 3.829461 | Best: 3.079980
Epoch  600 | Train: 0.693681 | Val: 2.519954 | Saved new best model
Epoch  700 | Train: 0.165212 | Val: 2.075793 | Saved new best model
Epoch  800 | Train: 0.124478 | Val: 2.160822 | Best: 2.075793
Epoch  900 | Train: 0.067973 | Val: 2.017642 | Saved new best model
Epoch 1000 | Train: 0.202080 | Val: 1.942103 | Saved new best model
Epoch 110

KeyboardInterrupt: 

Lastly, we plot the RMSE of test loss over epochs for different dataset sizes and training time (in the labels).

In [ ]:
# Prevents matplotlib from converting fonts to paths inside SVG
plt.rcParams['svg.fonttype'] = 'none'

# Height follows golden ratio to maintain clean proportions
fig, ax = plt.subplots(figsize=(single_col, single_col * golden_ratio))

# Convert to Python list because matplotlib colormaps expect native floats
color_indices = torch.linspace(0, 1, len(dataset_sizes)).tolist()
colors = [plt.cm.viridis(val) for val in color_indices]

# 3. Iterate Over Dataset Sizes
for i, n_samples in enumerate(dataset_sizes):

    # Construct file paths for stored history and training time
    hist_path = os.path.join(output_dir, f"history_{n_samples}.pt")
    time_path = os.path.join(output_dir, f"time_{n_samples}.txt")

    # Skip if training history does not exist
    if not os.path.exists(hist_path):
        continue

    # Load and format training time
    t_str = "N/A"
    if os.path.exists(time_path):
        with open(time_path, 'r') as f:
            total = float(f.read().strip())

        # Convert seconds into readable compact format
        if total > 3600:
            t_str = f"{total/3600:.1f}h"
        elif total > 60:
            t_str = f"{total/60:.1f}m"
        else:
            t_str = f"{total:.1f}s"

    data = torch.load(hist_path, map_location='cpu')

    if data['test']:
        epochs, mse = zip(*data['test'])

        # Convert MSE list to torch Tensor to compute sqrt
        rmse = torch.sqrt(torch.tensor(mse)).tolist()

        # Plot curve
        ax.plot(
            epochs,
            rmse,
            color=colors[i],
            label=f"{n_samples} ({t_str})"
        )

# Strict axis formatting
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Loss (RMSE)')

# Remove top/right spines for cleaner scientific aesthetic
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Compact legend (single-column friendly, no frame)
ax.legend(title="Samples (Time)", frameon=False)

# 7. Save as Editable SVG
save_path = os.path.join(output_dir, "nature_loss_comparison.svg")
plt.savefig(save_path, format="svg")

print(f"Figure saved to: {save_path}")
plt.show()

# Section 4: Neural Inverse Dynamics
In this section we derive equations of motion and solve the inverse dynamics problem numerically; the same problem is then solved with the pre-trained neural network. We then compare numerical and neural solutions.

In [ ]:
# Random seed for reproducible results
SEED = 23
torch.manual_seed(SEED)

# Number of simulated trajectories
NUM_TRAJECTORIES = 50

# Path for saving the generated dataset
SAVE_PATH = "double_pendulum_dataset.pt"

# Index of the trajectory to visualize
ID_TRAJ = 1

# Dataset sizes (number of samples)
MODEL_NUMS = dataset_sizes

# Base directory for inverse dynamics results
BASE_PATH = output_dir

# Fixed integration time step (seconds)
FIXED_STEP_SIZE = 1e-4

# Test trajectory duration, s
DURATION = 5

# Number of function evaluations (used for statistical testing)
n_repeats = 10000

First, we derive closed-form expressions for the double-pendulum accelerations using the Gibbs-Appell formulation, largely repeating procedures from one of the previous sections.

In [ ]:
# To simulate the double pendulum, we must derive explicit expressions
# for the generalized accelerations.

# Governing equation (Gibbs–Appell):
#     ∂S / ∂(ddq_i) = Q_i
#
# Right-hand side (generalized forces):
#     Q_i = −∂V / ∂q_i
#     Represents gravitational forces acting on each coordinate.
#
# Left-hand side (inertial terms):
#     ∂S / ∂(ddq_i)
#     Represents inertial resistance due to mass and motion.
#
# Dynamic equilibrium:
#     ∂S / ∂(ddq) = −∂V / ∂q
#     This system is solved symbolically for ddq.
# ==============================================================================

print("Deriving equations for simulation...")

# Define symbolic variables
t = sp.symbols('t')
m1, m2, l1, l2, g = sp.symbols('m1 m2 l1 l2 g')
q1, q2 = sp.symbols('q1 q2')
dq1, dq2 = sp.symbols('dq1 dq2')
ddq1, ddq2 = sp.symbols('ddq1 ddq2')

# Define generalized coordinates as time-dependent functions
theta1 = sp.Function('theta1')(t)
theta2 = sp.Function('theta2')(t)

# Map time-dependent variables to static symbols
subs_map = {
    theta1.diff(t, t): ddq1,
    theta2.diff(t, t): ddq2,
    theta1.diff(t): dq1,
    theta2.diff(t): dq2,
    theta1: q1,
    theta2: q2
}

# Re-derive expressions locally to avoid unintended symbolic coupling with
# variables from previous sections

# Mass positions
x1 = l1 * sp.sin(theta1)
y1 = -l1 * sp.cos(theta1)

x2 = x1 + l2 * sp.sin(theta1 + theta2)
y2 = y1 - l2 * sp.cos(theta1 + theta2)

# Gravitational potential energy
V_dynamic = m1 * g * y1 + m2 * g * y2
V_static = V_dynamic.subs(subs_map)

# Velocities
vx1, vy1 = sp.diff(x1, t), sp.diff(y1, t)
vx2, vy2 = sp.diff(x2, t), sp.diff(y2, t)

# Accelerations
ax1, ay1 = sp.diff(vx1, t), sp.diff(vy1, t)
ax2, ay2 = sp.diff(vx2, t), sp.diff(vy2, t)

# Gibbs–Appell function
S_symbolic = 0.5 * m1 * (ax1**2 + ay1**2) + 0.5 * m2 * (ax2**2 + ay2**2)
S_clean = S_symbolic.subs(subs_map).simplify()

# Generalized gravitational forces
Q_grav_1 = (-sp.diff(V_static, q1)).simplify()
Q_grav_2 = (-sp.diff(V_static, q2)).simplify()

# Inertial terms from Gibbs–Appell
tau1_sym = sp.diff(S_clean, ddq1).simplify()
tau2_sym = sp.diff(S_clean, ddq2).simplify()

# Solve the system for accelerations
print("Solving symbolic system...")
solution = sp.solve(
    [sp.Eq(tau1_sym, Q_grav_1),
     sp.Eq(tau2_sym, Q_grav_2)],
    [ddq1, ddq2]
)

ddq1_expr = solution[ddq1].simplify()
ddq2_expr = solution[ddq2].simplify()

print("Accelerations isolated symbolically.")

# Compile symbolic expressions into a Torch-compatible function
compute_accelerations = sp.lambdify(
    (q1, q2, dq1, dq2, m1, m2, l1, l2, g),
    (ddq1_expr, ddq2_expr),
    modules="torch"
)

print("Generalized acceleration ddq1:")
sp.pprint(ddq1_expr)

print("\nGeneralized acceleration ddq2:")
sp.pprint(ddq2_expr)


Second, we rewrite the double-pendulum equations in first-order form so they can be integrated numerically, returning time derivatives of angles and velocities at each step.

In [ ]:
# Numerical ODE solvers (e.g., Runge–Kutta) require the dynamics to be written
# as a first-order system of the form: d/dt [q, dq] = f(q, dq).

def derive_double_pendulum_ode(t, state, params):
    """
    Computes the time derivative of the system state for numerical integration.

    Args:
        t (float): Current simulation time (required by ODE solvers).
        state (Tensor): Current state [q1, q2, dq1, dq2].
        params (dict): Physical parameters (m1, m2, l1, l2).

    Returns:
        Tensor: Time derivatives [dq1, dq2, ddq1, ddq2].
    """

    # Convert physical parameters to tensors
    m1_t = torch.tensor(params['m1'])
    m2_t = torch.tensor(params['m2'])
    l1_t = torch.tensor(params['l1'])
    l2_t = torch.tensor(params['l2'])
    g_t  = torch.tensor(9.81)

    # Unpack the current state
    q1_v, q2_v  = state[0], state[1]
    dq1_v, dq2_v = state[2], state[3]

    # Compute generalized accelerations from symbolic dynamics
    accel_1, accel_2 = compute_accelerations(
        q1_v, q2_v, dq1_v, dq2_v,
        m1_t, m2_t, l1_t, l2_t, g_t
    )

    # Return the first-order state derivative
    return torch.tensor(
        [dq1_v, dq2_v, accel_1, accel_2],
        device=state.device
    )


Third, I define a function that generates double-pendulum trajectories by integrating the system's ODEs, recording angles, velocities, and accelerations at each timestep to serve as ground-truth data for later analyses.

In [ ]:
# A function to generate a single double-pendulum trajectory by numerically
# integrating the first-order ODE system using a fixed-step RK4 scheme.
def generate_trajectory(duration=5.0, dt=0.001):
    """
    Simulates the system from t = 0 to t = duration.

    Returns:
        Tensor: Trajectory snapshots [q1, q2, dq1, dq2, ddq1, ddq2] at each timestep.
        Tensor: Time axis corresponding to the trajectory.
    """

    # Create the time axis with a fixed step size
    t_axis = torch.arange(0.0, duration + dt, dt)

    # Initial conditions
    # Joint angles are sampled uniformly within a bounded range,
    # while initial angular velocities are set to zero.
    q_init  = torch.empty(2).uniform_(-torch.pi / 3, torch.pi / 3)
    dq_init = torch.zeros(2)

    # Assemble the initial state vector: [q1, q2, dq1, dq2]
    state = torch.cat([q_init, dq_init])

    traj = []

    # Time integration loop (RK4)
    for t in t_axis:

        # Evaluate the state derivative at the current time
        # This provides both velocities and accelerations.
        d_state = derive_double_pendulum_ode(t, state, PARAMS)
        ddq = d_state[2:]

        # Store the full snapshot: angles, velocities, and accelerations
        traj.append(torch.cat([state, ddq]))

        # Runge–Kutta 4 integration step
        # The next state is computed from a weighted average of four slopes.

        k1 = d_state
        k2 = derive_double_pendulum_ode(t + 0.5 * dt, state + 0.5 * dt * k1, PARAMS)
        k3 = derive_double_pendulum_ode(t + 0.5 * dt, state + 0.5 * dt * k2, PARAMS)
        k4 = derive_double_pendulum_ode(t + dt,        state + dt * k3,       PARAMS)

        state = state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    # Stack all snapshots into a single tensor of shape [T, 6]
    return torch.stack(traj), t_axis


If, instead of RK, you want to use semi-implicit Euler integration, uncommnet the cell below.

In [ ]:
# If you want to use implicit Euler and not Runge-Kutta solver for the ODE,
# uncomment this cell


# def generate_trajectory(duration=5.0, dt=0.001):
#     """
#     Simulates the system from t=0 to t=duration using Semi-Implicit Euler.
#     Returns the full dataset: [Position, Velocity, Acceleration] at each step.
#     """

#     # Create the time axis
#     # using arange to match step size exactly
#     t_axis = torch.arange(0.0, duration, dt)

#     # Initial Conditions
#     # q:  Uniform between -pi/2 and +pi/2
#     # dq: Uniform between -5 and +5 rad/s
#     q  = torch.empty(2).uniform_(-torch.pi/2, torch.pi/2)
#     dq = torch.zeros(2)

#     traj = []

#     # Integration Loop (Semi-Implicit Euler)
#     for t in t_axis:

#         # 1. Compute Dynamics (Acceleration)
#         # We temporarily pack state for the physics function
#         state_for_calc = torch.cat([q, dq])

#         d_state = derive_double_pendulum_ode(t, state_for_calc, PARAMS)
#         ddq = d_state[2:] # Extract acceleration

#         # 2. Store Snapshot: [q, dq, ddq]
#         # We store the state *before* the update to align input (q, dq) with target (ddq)
#         traj.append(torch.cat([q, dq, ddq]))

#         # 3. Semi-Implicit Euler Step
#         # Update Velocity FIRST
#         dq = dq + ddq * dt

#         # Update Position SECOND (using the new velocity)
#         q = q + dq * dt

#     # Stack the list of snapshots into a single Tensor [TimeSteps, 6]
#     return torch.stack(traj), t_axis

Next, we define a function that uses a trained neural network to compute generalized forces by differentiating the learned Gibbs-Appell function with respect to accelerations.

In [ ]:
def get_neural_torques(model, state_vector_norm, y_std, x_std):
    """
    Computes predicted torques from a trained Gibbs-Appell network.

    Args:
        model: Trained PyTorch model approximating S.
        state_vector_norm: Tensor [Batch, 6] of normalized [q, dq, ddq].
        y_std: Standard deviation of S (for rescaling).
        x_std: Standard deviation of input features (for rescaling).

    Returns:
        Tensor [Batch, 2]: Predicted generalized forces [tau1, tau2].
    """

    # Clone and prepare input for gradient computation
    state_input = state_vector_norm.clone().detach().requires_grad_(True)

    # Forward pass: predict normalized S
    s_pred_norm = model(state_input)

    # Compute gradients dS/d(state) using autograd
    grads = autograd.grad(
        outputs=s_pred_norm,
        inputs=state_input,
        grad_outputs=torch.ones_like(s_pred_norm),
        create_graph=True,
        only_inputs=True
    )[0]

    # Extract derivatives w.r.t. accelerations (ddq1, ddq2)
    d_ddq_norm = grads[:, 4:6]

    # Ensure input std is a tensor on the correct device
    if not isinstance(x_std, torch.Tensor):
        x_std = torch.tensor(x_std, dtype=state_input.dtype,
                             device=state_input.device)

    # Chain-rule scaling: convert normalized gradient to physical units
    ddq_std = x_std[4:6] + 1e-8  # avoid division by zero
    scale_factor = y_std / ddq_std

    # Apply scaling to get predicted torques in real units
    torque_pred = d_ddq_norm * scale_factor

    return torque_pred


Subsequently, we create a fixed test set to evaluate all networks consistently, either by loading a cached (previously generated and stored in file) dataset or generating new trajectories.

In [ ]:
# Setup file path for the fixed test set
TEST_SET_FILENAME = f"test_set_{NUM_TRAJECTORIES}_traj_{DURATION}s.pt"  # Include trajectory count & duration
test_set_path = os.path.join(BASE_PATH, TEST_SET_FILENAME)

# Pre-cast physical parameters for computation
m1_t = torch.tensor(PARAMS['m1'], device=device)
m2_t = torch.tensor(PARAMS['m2'], device=device)
l1_t = torch.tensor(PARAMS['l1'], device=device)
l2_t = torch.tensor(PARAMS['l2'], device=device)

# Load cached test set if it exists
if os.path.exists(test_set_path):
    print(f"Found cached test set at: {test_set_path}")
    print("Loading test set from file...")

    cached_data = torch.load(test_set_path, map_location='cpu')
    FIXED_INPUTS    = cached_data['inputs']
    FIXED_TARGETS   = cached_data['targets']
    fixed_time_axis = cached_data['time_axis']

    print(f"Loaded successfully. Shape: {FIXED_INPUTS.shape}")

# Otherwise, generate a new test set
else:
    print(f"Cache not found. Generating fixed test set with {NUM_TRAJECTORIES} trajectories...")

    fixed_inputs_list = []
    fixed_targets_list = []
    fixed_time_axis = None

    for i in range(NUM_TRAJECTORIES):
        print(f"Generating Trajectory {i+1}/{NUM_TRAJECTORIES}")

        traj_data, time_axis = generate_trajectory(duration=DURATION, dt=FIXED_STEP_SIZE)

        # Save time axis from the first trajectory
        if fixed_time_axis is None:
            fixed_time_axis = time_axis.cpu()

        # Move trajectory to GPU for torque calculation
        traj_data = traj_data.to(device)

        # Compute ground truth torques
        tau1_true, tau2_true = compute_tau_ground_truth(
            traj_data[:, 0], traj_data[:, 1],
            traj_data[:, 2], traj_data[:, 3],
            traj_data[:, 4], traj_data[:, 5],
            m1_t, m2_t, l1_t, l2_t
        )
        traj_true = torch.stack([tau1_true, tau2_true], dim=1)

        # Store CPU copies to save GPU memory
        fixed_inputs_list.append(traj_data.cpu())
        fixed_targets_list.append(traj_true.cpu())

    # Stack all trajectories into single tensors
    FIXED_INPUTS  = torch.stack(fixed_inputs_list)
    FIXED_TARGETS = torch.stack(fixed_targets_list)

    # Save for future reuse
    print(f"Saving generated test set to: {test_set_path}")
    torch.save({
        'inputs': FIXED_INPUTS,
        'targets': FIXED_TARGETS,
        'time_axis': fixed_time_axis
    }, test_set_path)

    # Optional Colab download
    if IN_COLAB: files.download(test_set_path)

    print(f"Test set ready. Shape: {FIXED_INPUTS.shape}")


Following that, we evaluate each trained model on the fixed test set by predicting torques, timing inference, and computing RMSE for both joints.

In [ ]:
# Instantiate the network once for evaluation
model = GibbsAppellNet().to(device)

# Evaluation loop for models trained on different dataset sizes
for num in MODEL_NUMS:
    print(f"\n{'='*40}")
    print(f"Processing Model: {num} samples")
    print(f"{'='*40}")

    model_path = os.path.join(BASE_PATH, f"model_{num}.pth")
    stats_path = os.path.join(BASE_PATH, f"stats_{num}.pt")
    save_name  = f"results_model_{num}.pt"
    full_save_path = os.path.join(BASE_PATH, save_name)

    # Skip if results already exist
    if os.path.exists(full_save_path):
        print(f"Skipping Model {num}: Results already exist ({save_name})")
        continue

    # Load normalization statistics for this model
    model_stats = torch.load(stats_path, map_location=device)
    stats_mean  = model_stats['X_mean'].to(device)
    stats_X_std = model_stats['X_std'].to(device)
    stats_S_std = model_stats['S_std'].to(device)

    # Load trained weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Containers for storing predictions, RMSE, and timing
    all_pred_tau_list = []
    inference_times   = []
    rmse_j1_list      = []
    rmse_j2_list      = []

    # Loop over fixed test trajectories
    for i in range(NUM_TRAJECTORIES):
        if (i+1) % 10 == 0:
            print(f"Evaluating Trajectory {i+1}/{NUM_TRAJECTORIES}")

        # Move data to GPU for inference
        traj_data = FIXED_INPUTS[i].to(device)
        traj_true = FIXED_TARGETS[i].to(device)

        # Normalize input using model-specific statistics
        input_norm = (traj_data - stats_mean) / (stats_X_std + 1e-8)

        # Time inference
        if device.type == 'cuda': torch.cuda.synchronize()
        t_start = time.perf_counter()

        t_pred = get_neural_torques(model, input_norm, stats_S_std, stats_X_std)

        if device.type == 'cuda': torch.cuda.synchronize()
        t_end = time.perf_counter()
        inference_times.append(t_end - t_start)

        # Store prediction
        all_pred_tau_list.append(t_pred.detach().cpu())

        # Compute RMSE for each joint
        sq_error = (traj_true - t_pred.detach()) ** 2
        rmse = torch.sqrt(torch.mean(sq_error, dim=0))
        rmse_j1_list.append(rmse[0].item())
        rmse_j2_list.append(rmse[1].item())

    # Save results for this model
    print(f"Saving results for model_{num}...")

    model_results = {
        str(FIXED_STEP_SIZE): {
            "inputs":        FIXED_INPUTS,
            "torques_true":  FIXED_TARGETS,
            "torques_pred":  torch.stack(all_pred_tau_list),
            "time_axis":     fixed_time_axis,
            "rmse_j1":       rmse_j1_list,
            "rmse_j2":       rmse_j2_list,
            "inference_times": inference_times,
            "dt":            FIXED_STEP_SIZE
        }
    }
    torch.save(model_results, full_save_path)

print("\nAll models processed on the same trajectories.")


Afterward, we benchmark the inference timing of models trained on different dataset sizes using repeated interleaved evaluation.

In [ ]:
# Output file for benchmark results
save_path = os.path.join(output_dir, "timing_benchmark_interleaved.pt")

# Flag to decide whether to run the benchmark
run_simulation = True

# Check if benchmark file exists and is valid
if os.path.exists(save_path):
    print(f"Found existing benchmark file at: {save_path}")
    try:
        checkpoint = torch.load(save_path)
        temp_data = checkpoint['benchmark_data']

        # Only use if data is already in dictionary format
        if isinstance(temp_data, dict):
            final_data = temp_data
            sizes_to_test = checkpoint['sizes_to_test']
            print("Data in correct format. Loading complete.")
            run_simulation = False
        else:
            print("WARNING: Old format detected. Re-running simulation...")
    except Exception as e:
        print(f"Error reading file ({e}). Re-running...")
else:
    print("No existing benchmark found. Starting new simulation...")

# Run benchmark if needed
if run_simulation:
    sizes_to_test = dataset_sizes
    benchmark_data = {size: [] for size in sizes_to_test}

    # Fix random seeds for reproducibility
    print(f"Setting random seed: {SEED}")
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Prepare dummy input for timing evaluation
    dummy_input = torch.rand(1, 6).to(device)
    dummy_input.requires_grad_(True)

    # Load all model weights and normalization stats into memory
    print("Loading models and stats...")
    model_pool = {}
    stats_pool = {}
    for size in sizes_to_test:
        model_pool[size] = torch.load(os.path.join(BASE_PATH, f"model_{size}.pth"), map_location=device)
        loaded_stats = torch.load(os.path.join(BASE_PATH, f"stats_{size}.pt"), map_location=device)
        stats_pool[size] = {'S_std': loaded_stats['S_std'].to(device),
                            'X_std': loaded_stats['X_std'].to(device)}

    # Instantiate network structure
    model = GibbsAppellNet().to(device)

    print(f"Running interleaved benchmark on {device}...")

    # Repeat timing measurements
    for i in range(n_repeats):
        if (i+1) % 500 == 0:
            print(f"Batch {i+1}/{n_repeats}")

        # Random order of dataset sizes each iteration
        indices = torch.randperm(len(sizes_to_test))
        for idx in indices:
            size = sizes_to_test[idx]

            # Load weights and set model to evaluation
            model.load_state_dict(model_pool[size])
            model.eval()

            stats_S_std = stats_pool[size]['S_std']
            stats_X_std = stats_pool[size]['X_std']

            # Measure inference time
            if device.type == 'cuda': torch.cuda.synchronize()
            t_start = time.perf_counter()
            _ = get_neural_torques(model, dummy_input, stats_S_std, stats_X_std)
            if device.type == 'cuda': torch.cuda.synchronize()
            t_end = time.perf_counter()

            benchmark_data[size].append((t_end - t_start) * 1000.0)  # ms

    # Convert timing lists to tensors
    final_data = {size: torch.tensor(benchmark_data[size]) for size in sizes_to_test}

    # Save benchmark results
    print(f"Saving benchmark results to: {save_path}")
    torch.save({'benchmark_data': final_data, 'sizes_to_test': sizes_to_test}, save_path)

    print("Benchmarking complete.")


### Measuring inference time

Next, we load or generate forward passes of the models with a dummy input to  measure inference time for each dataset size.

In [ ]:
# Check for existing data
run_simulation = True
if os.path.exists(save_path):
    print(f"Found existing benchmark file at: {save_path}")
    try:
        checkpoint = torch.load(save_path)
        temp_data = checkpoint['benchmark_data']

        # Validate that data is a dictionary
        if isinstance(temp_data, dict):
            final_data = temp_data
            sizes_to_test = checkpoint['sizes_to_test']
            print("Data is in correct format. Loading complete.")
            run_simulation = False
        else:
            print("Warning: existing data is in old format (list). Discarding and re-running...")
    except Exception as e:
        print(f"Error reading existing file ({e}). Re-running...")
else:
    print("No existing benchmark found. Starting new simulation...")

# Run simulation if needed
if run_simulation:
    sizes_to_test = dataset_sizes
    benchmark_data = {size: [] for size in sizes_to_test}

    # Set random seeds
    print(f"Setting random seed to: {SEED}")
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Create dummy input for benchmarking
    dummy_input = torch.rand(1, 6).to(device)
    dummy_input.requires_grad_(True)

    # Pre-load model weights and statistics
    print("Pre-loading models and stats into memory...")
    model_pool = {}
    stats_pool = {}

    for size in sizes_to_test:
        # Load model weights
        path_model = os.path.join(BASE_PATH, f"model_{size}.pth")
        model_pool[size] = torch.load(path_model, map_location=device)

        # Load normalization statistics
        path_stats = os.path.join(BASE_PATH, f"stats_{size}.pt")
        loaded_stats = torch.load(path_stats, map_location=device)

        stats_pool[size] = {
            'S_std': loaded_stats['S_std'].to(device),
            'X_std': loaded_stats['X_std'].to(device)
        }

    # Instantiate model once
    model = GibbsAppellNet().to(device)

    print(f"Starting interleaved benchmark on device: {device}...")

    for i in range(n_repeats):
        if (i + 1) % 500 == 0:
            print(f"Batch {i + 1}/{n_repeats}...")

        # Shuffle sizes to avoid ordering bias
        indices = torch.randperm(len(sizes_to_test))

        for idx in indices:
            size = sizes_to_test[idx]

            # Hot-swap model weights
            model.load_state_dict(model_pool[size])
            model.eval()

            # Select stats for this size
            stats_S_std = stats_pool[size]['S_std']
            stats_X_std = stats_pool[size]['X_std']

            # Measure inference time
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            # Forward pass through model
            _ = get_neural_torques(model, dummy_input, stats_S_std, stats_X_std)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()

            benchmark_data[size].append((t_end - t_start) * 1000.0)

    # Convert benchmark data to tensors
    final_data = {size: torch.tensor(benchmark_data[size]) for size in sizes_to_test}

    # Save benchmark results
    print(f"Saving new benchmark results to: {save_path}")
    torch.save({
        'benchmark_data': final_data,
        'sizes_to_test': sizes_to_test
    }, save_path)

    print("Benchmarking complete.")


Following this, we perform statistical analysis.

In [ ]:
import scipy.stats as scipy_stats

# Prepare a dataframe
data_records = []
for size, timings in final_data.items():
    timings_np = timings.cpu().numpy()
    for t in timings_np:
        data_records.append({'Dataset_Size': str(size), 'Time_ms': t})

df = pd.DataFrame(data_records)

print("\n" + "=" * 50)
print("Statistical analysis")
print("=" * 50)

# Normality test for each dataset size
print(f"\n{'Dataset Size':<15} | {'Shapiro-Wilk p-value':<20} | {'Result'}")
print("-" * 55)

is_normal = True
grouped_data = []
sorted_sizes = sorted(sizes_to_test)

for size in sorted_sizes:
    times = df[df['Dataset_Size'] == str(size)]['Time_ms']
    grouped_data.append(times)

    stat, p_value = scipy_stats.shapiro(times)
    is_distribution_normal = p_value > 0.05
    if not is_distribution_normal:
        is_normal = False

    res_str = "Normal" if is_distribution_normal else "Not normal"
    print(f"{str(size):<15} | {p_value:.4e}{' '*10} | {res_str}")


# Perform significance test across dataset sizes
print("\n--- Hypothesis testing ---")
if is_normal:
    stat, p_val = scipy_stats.f_oneway(*grouped_data)
    test_name = "One-way ANOVA"
else:
    stat, p_val = scipy_stats.kruskal(*grouped_data)
    test_name = "Kruskal-Wallis H-test"

print(f"Test performed: {test_name}")
print(f"Statistic: {stat:.4f} | P-Value: {p_val:.4e}")

if p_val < 0.05:
    print(">> Result: Significant difference detected (reject H0).")
else:
    print(">> Result: No significant difference (fail to reject H0).")


# Compute descriptive statistics for each dataset size
print("\n--- Descriptive statistics ---")
print(f"{'Dataset Size':<15} | {'N':<6} | {'Mean ± Std (ms)':<20} | {'Median [IQR] (ms)':<20} | {'Min / Max'}")
print("-" * 95)

for size in sorted_sizes:
    group_data = df[df['Dataset_Size'] == str(size)]['Time_ms']

    n       = len(group_data)
    mean    = group_data.mean()
    std     = group_data.std()
    q1      = group_data.quantile(0.25)
    median  = group_data.median()
    q3      = group_data.quantile(0.75)
    iqr     = q3 - q1
    min_val = group_data.min()
    max_val = group_data.max()

    mean_std_str   = f"{mean:.3f} ± {std:.3f}"
    median_iqr_str = f"{median:.3f} [{iqr:.3f}]"
    min_max_str    = f"{min_val:.3f} / {max_val:.3f}"

    print(f"{str(size):<15} | {n:<6} | {mean_std_str:<20} | {median_iqr_str:<20} | {min_max_str}")

print("-" * 95)


# Compute global statistics across all dataset sizes
all_times     = df['Time_ms']
global_mean   = all_times.mean()
global_std    = all_times.std()
global_median = all_times.median()

print(f"Overall (all sizes combined):")
print(f"Mean ± Std:    {global_mean:.4f} ± {global_std:.4f} ms")
print(f"Median:        {global_median:.4f} ms")
print("-" * 95 + "\n")

Finally, we generate a figure of inference speed across dataset sizes using Nature style settings.


In [ ]:
save_path        = os.path.join(output_dir, "timing_benchmark_interleaved.pt")
figure_save_path = os.path.join(output_dir, "nature_inference_speed_benchmark.svg")

# Apply the settings globally
single_col, double_col, golden_ratio = set_nature_style()
print("Nature Communications style applied.")

# Use double column width for the plot
fig_width  = double_col
fig_height = double_col * golden_ratio * 0.75  # Adjust height slightly for better aspect ratio in paper

plt.figure(figsize=(fig_width, fig_height))

# Enable y-axis grid with subtle lines (optional in Nature)
plt.grid(True, axis='y', linestyle='--', alpha=0.5, linewidth=0.5)

# Create boxplot using seaborn
ax = sns.boxplot(
    x='Dataset_Size',
    y='Time_ms',
    hue='Dataset_Size',
    data=df,
    order=[str(s) for s in sorted_sizes],
    palette=['#4DBBD5', '#E64B35', '#00A087', '#3C5488'],
    showfliers=False,
    width=0.5,
    boxprops=dict(alpha=0.8, linewidth=0.5),
    whiskerprops=dict(linewidth=0.5),
    capprops=dict(linewidth=0.5),
    medianprops=dict(linewidth=0.8, color='black'),
    legend=False
)

# Set axis labels and title
plt.title('Inference Speed vs Training Dataset Size')
plt.ylabel('Inference Time (ms)')
plt.xlabel('Training Dataset Size')

# Adjust layout for better fit
plt.tight_layout()

# Save figure as high-resolution PDF
print(f"Saving figure to: {figure_save_path}")
plt.savefig(figure_save_path, dpi=600)  # Explicit dpi for publication quality

# Display the figure
plt.show()


### Measuring prediction accuracy

First, we load RMSE results from saved model outputs and organize them into a DataFrame.


In [ ]:
# Configuration
results_path  = BASE_PATH
sorted_sizes  = sorted(MODEL_NUMS)  # Ensure dataset sizes are processed in ascending order

print(f"Loading RMSE results from: {results_path}")
records = []

for size in sorted_sizes:
    filename  = f"results_model_{size}.pt"
    full_path = os.path.join(results_path, filename)

    # Skip if file does not exist
    if not os.path.exists(full_path):
        print(f"Warning: {filename} not found.")
        continue

    # Load the saved results
    data = torch.load(full_path)

    # Extract the inner dict (grab the first available key, e.g., step size)
    key = list(data.keys())[0]
    res = data[key]

    # Unpack RMSE lists for each joint
    rmse_j1 = res['rmse_j1']
    rmse_j2 = res['rmse_j2']

    # Append each value to records for the DataFrame
    for val in rmse_j1:
        records.append({'Dataset_Size': size, 'Joint': 'J1', 'RMSE': val})
    for val in rmse_j2:
        records.append({'Dataset_Size': size, 'Joint': 'J2', 'RMSE': val})

# Create a pandas DataFrame from all records
df_rmse = pd.DataFrame(records)
print(f"Data loaded. DataFrame shape: {df_rmse.shape}")


Second, we perform a Shapiro-Wilk normality test on RMSE values for each joint and dataset size

In [ ]:
import scipy.stats as scipy_stats

print(f"\n{'='*20} Normality Test (Shapiro-Wilk) {'='*20}")
print(f"{'Joint':<6} | {'Dataset Size':<12} | {'P-Value':<12} | {'Distribution'}")
print("-" * 60)

# Dictionary to store normality results
normality_results = {}

for joint in ['J1', 'J2']:
    for size in sorted_sizes:

        # Extract RMSE values for this joint and dataset size
        subset = df_rmse[(df_rmse['Dataset_Size'] == size) & (df_rmse['Joint'] == joint)]['RMSE']

        # Perform Shapiro-Wilk test
        stat, p = scipy_stats.shapiro(subset)
        is_normal = p > 0.05
        normality_results[(joint, size)] = is_normal

        # Prepare distribution label
        dist_str = "Normal" if is_normal else "Not Normal"

        # Highlight significant deviations with asterisk
        p_str = f"{p:.2e}"
        if not is_normal:
            p_str = f"*{p_str}"

        # Print results in a table-like format
        print(f"{joint:<6} | {str(size):<12} | {p_str:<12} | {dist_str}")

print("-" * 60)
print("* p < 0.05 indicates data is not normally distributed.")

Third, we calculate and plot pairwise significance matrices (p-values) for RMSE comparisons across dataset sizes.


In [ ]:
import scipy.stats as scipy_stats

def plot_significance_matrix(joint_name, df, sizes, save_path=None, correction='fdr_bh'):
    """Calculate, plot, and print explanations for an FDR-corrected p-value matrix."""

    n = len(sizes)
    p_matrix = np.zeros((n, n))

    # Lists to store the raw p-values and their matrix indices
    raw_p_values = []
    indices = []

    # 1. Calculate pairwise raw p-values (upper triangle only to avoid duplicates)
    for i in range(n):
        for j in range(i + 1, n):
            size_a = sizes[i]
            size_b = sizes[j]

            data_a = df[(df['Dataset_Size'] == size_a) & (df['Joint'] == joint_name)]['RMSE']
            data_b = df[(df['Dataset_Size'] == size_b) & (df['Joint'] == joint_name)]['RMSE']

            stat, p = scipy_stats.mannwhitneyu(data_a, data_b, alternative='two-sided')
            raw_p_values.append(p)
            indices.append((i, j))

    # 2. Apply Benjamini-Hochberg (FDR) multiple comparison correction
    reject, corrected_p_values, _, _ = multipletests(raw_p_values, alpha=0.05, method=correction)

    # 3. Reconstruct the symmetric matrix with corrected p-values
    p_matrix[:] = np.nan # Fill with NaN initially (masks diagonal)
    for k, (i, j) in enumerate(indices):
        p_matrix[i, j] = corrected_p_values[k]
        p_matrix[j, i] = corrected_p_values[k] # Mirror for lower triangle

    # Set up figure
    fig_width = single_col  # Fit in single column width
    plt.figure(figsize=(fig_width, fig_width))  # Square aspect ratio

    mask = np.eye(n, dtype=bool)  # Mask diagonal

    # 4. Create discrete colormap: White for p < 0.05, Red for p >= 0.05
    cmap = mcolors.ListedColormap(['black', 'red'])
    bounds = [0.0, 0.05, 1.0]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    # Plot heatmap
    ax = sns.heatmap(
        p_matrix,
        annot=True,
        fmt='.1e',
        mask=mask,
        xticklabels=sizes,
        yticklabels=sizes,
        cmap=cmap,
        norm=norm,             # Apply the strict 0.05 boundary
        cbar_kws={'label': 'FDR Adj. p-value', 'shrink': 0.8, 'ticks': [0.025, 0.525]},
        annot_kws={"size": 5}  # Smaller text for annotations
    )

    # Clean up the colorbar ticks to make it clear what the colors mean
    colorbar = ax.collections[0].colorbar
    colorbar.set_ticklabels(['Significant\n(p < 0.05)', 'Not Sig.\n(p ≥ 0.05)'])

    # Set titles and labels
    plt.title(f'{joint_name}: Pairwise Differences (FDR Corrected)')
    plt.xlabel('Dataset Size A')
    plt.ylabel('Dataset Size B')

    # Save figure if path is provided
    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches='tight')

    plt.show()

    # ---------------------------------------------------------
    # 5. Generate Publication-Ready Text Explanation
    # ---------------------------------------------------------
    print(f"\n--- Statistical Summary for Joint {joint_name} ---")
    print(f"To assess the impact of dataset size on model performance for joint {joint_name}, "
          f"pairwise Mann-Whitney U tests were performed on the RMSE distributions.")
    print(f"To control the false discovery rate associated with multiple comparisons ({len(raw_p_values)} tests), "
          f"p-values were adjusted using the Benjamini-Hochberg procedure with a significance threshold of \u03B1 = 0.05.")

    sig_pairs = []
    nonsig_pairs = []
    for k, (i, j) in enumerate(indices):
        pair_str = f"{sizes[i]} vs {sizes[j]}"
        if reject[k]:
            sig_pairs.append(pair_str)
        else:
            nonsig_pairs.append(pair_str)

    if len(nonsig_pairs) == 0:
        print("The analysis revealed that all pairwise comparisons between different dataset sizes "
              "yielded statistically significant differences in RMSE (FDR-adjusted p < 0.05). "
              "This indicates that every tested increase in dataset size resulted in a measurable "
              "change in model performance.")
    elif len(sig_pairs) == 0:
        print("The analysis revealed no statistically significant differences in RMSE between any of "
              "the dataset sizes (FDR-adjusted p \u2265 0.05).")
    else:
        print(f"Statistically significant differences in RMSE (FDR-adjusted p < 0.05) were found for "
              f"the following comparisons: {', '.join(sig_pairs)}.")
        print(f"However, no significant differences were observed between: {', '.join(nonsig_pairs)}.")
    print("-" * 50 + "\n")


print(f"\n{'='*20} Pairwise Statistical Matrices {'='*20}")

# Plot and save for joint J1
path_j1 = os.path.join(results_path, "significance_matrix_j1.pdf")
plot_significance_matrix('J1', df_rmse, sorted_sizes, save_path=path_j1)

# Plot and save for joint J2
path_j2 = os.path.join(results_path, "significance_matrix_j2.pdf")
plot_significance_matrix('J2', df_rmse, sorted_sizes, save_path=path_j2)

Lastly, we create box plots to visualize RMSE distribution across dataset sizes for each joint.

In [ ]:
# Calculate and print descriptive statistics for RMSE
print(f"\n{'='*20} Descriptive Statistics (RMSE) {'='*20}")
print(f"{'Joint':<6} | {'Dataset Size':<12} | {'Mean ± Std':<20} | {'Median [IQR]':<20}")
print("-" * 70)

# Ensure sorting for consistent output
joints = sorted(df_rmse['Joint'].unique())
sizes  = sorted(df_rmse['Dataset_Size'].unique(), key=int)

# Following this, we loop through each joint and dataset size to compute descriptive metrics
for joint in joints:
    for size in sizes:
        # Filter the data for the current joint and dataset size
        subset = df_rmse[(df_rmse['Dataset_Size'] == size) & (df_rmse['Joint'] == joint)]['RMSE']
        if len(subset) == 0:
            continue

        # Compute mean, standard deviation, median, and interquartile range
        mean_val = subset.mean()
        std_val  = subset.std()
        median   = subset.median()
        q1       = subset.quantile(0.25)
        q3       = subset.quantile(0.75)
        iqr      = q3 - q1

        # Format metrics for clean printing
        mean_std = f"{mean_val:.4f} ± {std_val:.4f}"
        med_iqr  = f"{median:.4f} [{iqr:.4f}]"

        print(f"{joint:<6} | {str(size):<12} | {mean_std:<20} | {med_iqr:<20}")

print("-" * 70 + "\n")


# Generate and save box plot for RMSE distribution

# Following this, we create a Nature-style box plot for RMSE across dataset sizes

# Define file path for saving figure
fig_name = "nature_rmse_distribution_boxplot.svg"
save_fig_path = os.path.join(results_path, fig_name)

# Set figure dimensions for double-column width (Nature style)
fig_width  = double_col
fig_height = double_col * golden_ratio * 0.8  # Slightly shorter for compactness
plt.figure(figsize=(fig_width, fig_height))

# Create box plot with joint-specific colors and no outliers
ax = sns.boxplot(
    data=df_rmse,
    x='Dataset_Size',
    y='RMSE',
    hue='Joint',
    palette=['#4DBBD5', '#E64B35'],  # Nature-style blue/red
    showfliers=False,                 # Hide outliers for cleaner view
    width=0.6,
    linewidth=0.8
)

# Set plot title and axis labels
plt.title('RMSE Distribution by Training Dataset Size')
plt.ylabel('Root Mean Squared Error (RMSE)')
plt.xlabel('Training Dataset Size (Samples)')

# Configure legend without frame
plt.legend(title='Joint', loc='upper right', frameon=False)

# Add light horizontal grid lines for clarity
plt.grid(True, axis='y', linestyle='--', alpha=0.5, linewidth=0.5)

# Adjust layout, save figure at high resolution, and display
plt.tight_layout()
plt.savefig(save_fig_path, dpi=600)
print(f"Figure saved to: {save_fig_path}")

plt.show()


### Supplementary

As a cherry on the top, here, we load saved model predictions for a fixed test trajectory and compare predicted joint torques across training dataset sizes against the same ground truth over time.

In [ ]:
# Configuration
MODEL_FOLDER = BASE_PATH
RESULT_FILES = [
    'results_model_1000.pt',
    'results_model_5000.pt',
    'results_model_10000.pt',
    'results_model_50000.pt'
]
STEP_SIZE_KEY = "0.0001"
TRIAL_IDX     = 0

colors      = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
line_styles = ['--', '--', '--', '--']

# Initialize containers
ground_truth      = None
time_axis         = None
model_predictions = {}

print(f"Loading data for trial #{TRIAL_IDX}...")

# Load prediction and ground-truth data for each trained model
for i, res_file in enumerate(RESULT_FILES):
    load_path = os.path.join(MODEL_FOLDER, res_file)

    if not os.path.exists(load_path):
        print(f"Skipping {res_file}, file not found.")
        continue

    try:
        checkpoint = torch.load(load_path, map_location='cpu')

        # Extract results for the selected integration step size
        if 'results_by_step_size' in checkpoint:
            data = checkpoint['results_by_step_size'][STEP_SIZE_KEY]
        else:
            data = checkpoint[STEP_SIZE_KEY]

        # Create a clean label from the dataset size
        raw_num = res_file.replace('results_model_', '').replace('.pt', '').replace('.pth', '')
        label   = f"N={int(raw_num):,}"

        # Extract predictions, ground truth, and time axis
        curr_pred = data['torques_pred'][TRIAL_IDX]
        curr_true = data['torques_true'][TRIAL_IDX]
        curr_time = data['time_axis']

        # Sanity check consistency of ground truth across models
        if ground_truth is not None:
            if not torch.allclose(curr_true, ground_truth, atol=1e-4):
                print(f"Warning: Ground truth mismatch in {res_file}.")

        model_predictions[label] = curr_pred

        # Store ground truth and time axis once
        if ground_truth is None:
            ground_truth = curr_true
            time_axis    = curr_time
            print(f"Loaded: {label}")

    except Exception as e:
        print(f"Error loading {res_file}: {e}")

# Plot results using Nature-style formatting
if ground_truth is None:
    print("No valid data found.")
else:
    print("Plotting results with Nature style...")

    # Use single-column width with increased height for stacked subplots
    fig_width  = single_col
    fig_height = single_col * golden_ratio * 1.8

    fig, axes = plt.subplots(2, 1, figsize=(fig_width, fig_height), sharex=True)

    # Plot torque 1
    ax1 = axes[0]
    ax1.plot(
        time_axis,
        ground_truth[:, 0],
        color='black',
        linewidth=1.2,
        alpha=0.6,
        label='Ground truth'
    )

    for idx, (label, pred) in enumerate(model_predictions.items()):
        ax1.plot(
            time_axis,
            pred[:, 0],
            color=colors[idx % len(colors)],
            linestyle=line_styles[idx % len(line_styles)],
            linewidth=0.8,
            label=label
        )

    ax1.set_ylabel('Torque 1 (Nm)')
    ax1.legend(loc='best', frameon=False, labelspacing=0.2)
    ax1.grid(True, alpha=0.3, linewidth=0.3)

    # Plot torque 2
    ax2 = axes[1]
    ax2.plot(
        time_axis,
        ground_truth[:, 1],
        color='black',
        linewidth=1.2,
        alpha=0.6,
        label='Ground truth'
    )

    for idx, (label, pred) in enumerate(model_predictions.items()):
        ax2.plot(
            time_axis,
            pred[:, 1],
            color=colors[idx % len(colors)],
            linestyle=line_styles[idx % len(line_styles)],
            linewidth=0.8,
            label=label
        )

    ax2.set_ylabel('Torque 2 (Nm)')
    ax2.set_xlabel('Time (s)')
    ax2.grid(True, alpha=0.3, linewidth=0.3)

    # Final layout adjustments
    plt.tight_layout()
    fig.align_ylabels(axes)

    # Define filename
    save_filename = "nature_torque_trajectory_comparison"

    # Save as PDF (Vector - Best for Nature submission/Zooming)
    plt.savefig(f"{save_filename}.pdf")

    # Save as PNG (Raster - High Res based on your dpi=600 setting)
    plt.savefig(f"{save_filename}.png")

    print(f"Figures saved as {save_filename}.pdf and {save_filename}.png")

    plt.show()

# Section 5: Neural Forward Dynamics

In the previous section, we have solved inverse dynamics problem with differently-trained neural networks, and selected one network as the best suited for the task. In this section, we show that this same network can be used to solve forward dynamics problem (no re-training needed).

### Constants

In [ ]:
# Dataset size the network was trained on
N_SAMPLES = 10000

# Base directory for loading/saving files
BASE_PATH = "."

# Fix random seed for reproducibility
SEED = 23
torch.manual_seed(SEED)

# Number of trajectories to simulate
NUM_TRAJS = 50

# Simulation time step
DT_SIM = 0.002

# Total simulation duration
DURATION = 5.0

# Total number of integration steps
STEPS = int(DURATION / DT_SIM)

# Discrete time vector for simulation
T_SPAN = torch.linspace(0, DURATION, STEPS)

# Filename for saving the full simulation dataset
save_filename = 'simulation_results_50trials.pt'

# File where timing statistics will be stored
timing_filename = "timing_stats.pt"


### Derivations

First, to avoid propagation of numeric variables from the previous sections, we need to RE-define the symbolic variables and functions that describe the system and its motion over time.

In [ ]:
# Define time
t = sp.symbols('t')

# Define system parameters
m1, m2 = sp.symbols('m1 m2')
l1, l2 = sp.symbols('l1 l2')
g      = sp.symbols('g')

# Define generalized coordinates as functions of time
theta1 = sp.Function('theta1')(t)
theta2 = sp.Function('theta2')(t)

# Define substitution symbols for positions, velocities, and accelerations
q1, q2     = sp.symbols('q1 q2')
dq1, dq2   = sp.symbols('dq1 dq2')
ddq1, ddq2 = sp.symbols('ddq1 ddq2')

print("Symbols and coordinates defined.")


# Compute position of mass 1
x1 = l1 * sp.sin(theta1)
y1 = -l1 * sp.cos(theta1)

# Compute position of mass 2 relative to link 1
x2 = x1 + l2 * sp.sin(theta1 + theta2)
y2 = y1 - l2 * sp.cos(theta1 + theta2)

print("Position vectors defined.")


# Compute velocities as first derivatives of positions
vx1 = sp.diff(x1, t)
vy1 = sp.diff(y1, t)
vx2 = sp.diff(x2, t)
vy2 = sp.diff(y2, t)

# Compute accelerations as second derivatives of positions
ax1 = sp.diff(vx1, t)
ay1 = sp.diff(vy1, t)
ax2 = sp.diff(vx2, t)
ay2 = sp.diff(vy2, t)

print("Velocity and acceleration vectors defined.")


# Compute squared magnitudes of accelerations
accel_mag_sq_1 = ax1**2 + ay1**2
accel_mag_sq_2 = ax2**2 + ay2**2

# Compute Gibbs-Appell function S = 0.5 * sum(m_i * a_i^2)
S_symbolic = 0.5 * m1 * accel_mag_sq_1 + 0.5 * m2 * accel_mag_sq_2

print("Gibbs-Appell function derived.")


Second, we again replace time-dependent derivatives and generalized coordinates with static symbols to make the final expressions easier to read and work with.

In [ ]:
# Create a mapping from dynamic functions to static symbols
# Derivatives must be substituted before the base functions
subs_map = {
    theta1.diff(t, t): ddq1,  # acceleration of joint1
    theta2.diff(t, t): ddq2,  # acceleration of joint2
    theta1.diff(t):    dq1,   # velocity of joint1
    theta2.diff(t):    dq2,   # velocity of joint2
    theta1:             q1,   # angle of joint1
    theta2:             q2    # angle of joint2
}

# Apply the substitutions and simplify the expression
S_clean = S_symbolic.subs(subs_map).simplify()

# Display the simplified Gibbs-Appell function
print("Gibbs-Appell Function:")
sp.pprint(S_clean)


Third, we symbolically derive the exact equations of motion and convert them into differentiable numerical functions.

In [ ]:
# Extract M(q) and bias terms from the symbolic action S.
# By construction, dS/d(ddq) gives the inertial forces:
# M(q) * ddq + C(q, dq)

# Compute symbolic inertial force components
inertial_force_1 = sp.diff(S_clean, ddq1)
inertial_force_2 = sp.diff(S_clean, ddq2)

# Extract mass matrix entries as partial derivatives w.r.t. accelerations
M11_s = sp.diff(inertial_force_1, ddq1)
M12_s = sp.diff(inertial_force_1, ddq2)
M21_s = sp.diff(inertial_force_2, ddq1)
M22_s = sp.diff(inertial_force_2, ddq2)

# Compute Coriolis and centrifugal bias by setting accelerations to zero
C_bias_1 = inertial_force_1.subs({ddq1: 0, ddq2: 0})
C_bias_2 = inertial_force_2.subs({ddq1: 0, ddq2: 0})

# Compute gravity vector from the potential energy
# Function-valued coordinates are replaced with scalar symbols for differentiation
y1_scalar = y1.subs({theta1: q1})
y2_scalar = y2.subs({theta1: q1, theta2: q2})

V_sym = m1 * g * y1_scalar + m2 * g * y2_scalar
G1_s  = sp.diff(V_sym, q1)
G2_s  = sp.diff(V_sym, q2)

# Create numerical functions for mass matrix, bias, and gravity terms
calc_M_robust = sp.lambdify(
    (q1, q2, m1, m2, l1, l2),
    [[M11_s, M12_s], [M21_s, M22_s]],
    'torch'
)

calc_C_robust = sp.lambdify(
    (q1, q2, dq1, dq2, m1, m2, l1, l2),
    [C_bias_1, C_bias_2],
    'torch'
)

calc_G_robust = sp.lambdify(
    (q1, q2, m1, m2, l1, l2, g),
    [G1_s, G2_s],
    'torch'
)

print("Symbolic derivation complete.")

Fourth, we define the system's ODE for solving the resulting equations of motion for joint accelerations.

In [ ]:
def derive_ode_robust(y, t, m1, m2, l1, l2, g_val):

    # Unpack state vector (keep everything as Tensors)
    theta1_val, theta2_val, dtheta1_val, dtheta2_val = y

    # Mass matrix M (2x2)
    # calc_M_robust returns a list-of-lists → stack rows → stack matrix
    M_list = calc_M_robust(theta1_val, theta2_val, m1, m2, l1, l2)
    M = torch.stack([torch.stack(row) for row in M_list])

    # Coriolis vector C (ensure flat 1D Tensor)
    C_list = calc_C_robust(theta1_val, theta2_val,
                           dtheta1_val, dtheta2_val,
                           m1, m2, l1, l2)
    if isinstance(C_list[0], list):          # handle [[c1], [c2]]
        C = torch.stack([item[0] for item in C_list])
    else:
        C = torch.stack(C_list)

    # Gravity vector G (same flattening logic as C)
    G_list = calc_G_robust(theta1_val, theta2_val,
                           m1, m2, l1, l2, g_val)
    if isinstance(G_list[0], list):
        G = torch.stack([item[0] for item in G_list])
    else:
        G = torch.stack(G_list)

    # Solve dynamics: M * ddq = -(C + G)
    RHS = -(C + G)
    ddq = torch.linalg.solve(M, RHS)

    # 4. Return full state derivative
    # dy/dt = [dtheta1, dtheta2, ddq1, ddq2]
    return torch.cat([y[2:], ddq])


print("Torch numerical function created.")



Next, we load statistics of the dataset used to train the model.

In [ ]:
# Construct the path to the saved normalization statistics file
stats_path = os.path.join(BASE_PATH, f"stats_{N_SAMPLES}.pt")

# Load statistics and map all tensors to CPU
stats_data = torch.load(stats_path, map_location='cpu')

def ensure_tensor(val):
    """
    Ensure the input is a torch.Tensor.
    """
    if torch.is_tensor(val):
        return val
    # Convert standard Python types to torch.Tensor
    return torch.tensor(val)

# Extract scalar normalization statistics
# Keeping these as 0-d tensors (scalars) allows for easy broadcasting later
S_mean = ensure_tensor(stats_data['S_mean'])
S_std  = ensure_tensor(stats_data['S_std'])

# Extract vector-valued normalization statistics
X_mean           = ensure_tensor(stats_data['X_mean'])
X_std            = ensure_tensor(stats_data['X_std'])
chain_rule_scale = ensure_tensor(stats_data['chain_scale'])

# Print loaded statistics for verification
print(f"Stats loaded from: {stats_path}")
print("-" * 30)

# Print scalar values using .item() for readability
print(f"S_mean:           {S_mean.item():.6f} (Type: {type(S_mean)})")
print(f"S_std:            {S_std.item():.6f} (Type: {type(S_std)})")

# Print vector/tensor statistics
print(f"Chain rule scale: {chain_rule_scale} (Shape: {chain_rule_scale.shape})")
print(f"X_mean shape:     {X_mean.shape}")
print(f"X_std shape:      {X_std.shape}")


### Optimization

Following this, we code up a function that solves for the joint accelerations $\ddot{\mathbf{q}} \in \mathbb{R}^2$ by minimizing a discretized formulation of Gauss's principle of least constraint. The objective function $J(\ddot{\mathbf{q}})$ combines the learned Gibbs-Appell function $S_{\text{net}}$, the virtual work done by gravity $\mathbf{G}(\mathbf{q})$, and a Tikhonov-style regularization term:

$$
\ddot{\mathbf{q}}^* = \operatorname*{argmin}_{\ddot{\mathbf{q}}} \left( S_{\text{net}}(\mathbf{q}, \dot{\mathbf{q}}, \ddot{\mathbf{q}}) + \mathbf{G}(\mathbf{q})^\top \ddot{\mathbf{q}} + \lambda \|\ddot{\mathbf{q}} - \ddot{\mathbf{q}}_{\text{prev}}\|^2 \right)
$$

We perform the optimization in two stages: (1) a coarse global search using random sampling ($N=512$) to identify a robust initial guess, followed by (2) a gradient-based local refinement (SLSQP) to converge on the precise minimizer.

In [ ]:
def neural_optimizer(
    q_in, dq_in, model, x_mean, x_std, s_mean, s_std, chain_scale, prev_ddq_guess=None
):
    """
    Finds the joint acceleration 'ddq' that minimizes the Gauss cost:
    J = S_net(q, dq, ddq) + G(q) · ddq + Regularization
    """

    # Prepare tensors

    # Initialize previous guess if not provided
    if prev_ddq_guess is None:
        prev_ddq_guess = np.zeros(2)

    # Convert numpy arrays to float32 tensors
    x_mean_t = torch.tensor(x_mean, dtype=torch.float32)
    x_std_t  = torch.tensor(x_std, dtype=torch.float32)
    s_mean_t = torch.tensor(s_mean, dtype=torch.float32)
    s_std_t  = torch.tensor(s_std, dtype=torch.float32)

    q_t       = torch.tensor(q_in, dtype=torch.float32)
    dq_t      = torch.tensor(dq_in, dtype=torch.float32)
    prev_ddq_t = torch.tensor(prev_ddq_guess, dtype=torch.float32)

    # Compute gravity vector for current state
    G_vec_np = np.array(
        calc_G_robust(
            q_in[0], q_in[1],
            PARAMS['m1'], PARAMS['m2'],
            PARAMS['l1'], PARAMS['l2'], 9.81
        )
    )
    G_vec_t = torch.tensor(G_vec_np, dtype=torch.float32)

    # Phase 1: Coarse search with vectorized candidates
    N_SAMPLES  = 512
    bounds_val = 30.0

    # Generate candidate accelerations
    candidates = torch.empty(N_SAMPLES, 2).uniform_(-bounds_val, bounds_val)
    candidates[0] = torch.zeros(2)       # Stop guess
    candidates[1] = prev_ddq_t           # Previous acceleration guess

    # Expand q and dq to match candidate batch size
    q_batch  = q_t.expand(N_SAMPLES, 2)
    dq_batch = dq_t.expand(N_SAMPLES, 2)
    X_batch  = torch.cat([q_batch, dq_batch, candidates], dim=1)

    # Normalize batch inputs
    X_batch_norm = (X_batch - x_mean_t) / (x_std_t + 1e-8)

    # Perform batch inference
    with torch.no_grad():
        S_pred_norm = model(X_batch_norm).flatten()

    # Denormalize predicted S
    S_real = S_pred_norm * s_std_t + s_mean_t

    # Compute physics-based cost (S + work + regularization)
    work_term = torch.sum(candidates * G_vec_t, dim=1)
    LAMBDA_REG = 0.1
    reg_term   = LAMBDA_REG * torch.sum((candidates - prev_ddq_t)**2, dim=1)

    costs = S_real + work_term + reg_term

    # Select best candidate
    best_idx   = torch.argmin(costs)
    best_guess = candidates[best_idx]

    # Phase 2: Fine tuning via PyTorch gradient descent

    # Initialize optimizer with best candidate
    ddq_opt = best_guess.clone().requires_grad_(True)
    optimizer = torch.optim.Adam([ddq_opt], lr=0.1)

    for _ in range(15):  # Typically enough steps for convergence
        optimizer.zero_grad()

        # Reconstruct input vector: [q, dq, ddq_opt]
        x_in = torch.cat([q_t, dq_t, ddq_opt], dim=0)
        x_norm = (x_in - x_mean_t) / (x_std_t + 1e-8)

        # Forward pass
        s_val_norm = model(x_norm.unsqueeze(0)).squeeze()

        # Denormalize S
        s_val_real = s_val_norm * s_std_t + s_mean_t

        # Physics loss + smoothness regularization
        phys_loss   = s_val_real + torch.dot(G_vec_t, ddq_opt)
        smooth_loss = LAMBDA_REG * torch.sum((ddq_opt - prev_ddq_t)**2)

        loss = phys_loss + smooth_loss
        loss.backward()
        optimizer.step()

    # Return optimized accelerations as numpy array
    return ddq_opt.detach().numpy()


Next, we code up the initialization of the model on the selected device, set the it to evaluation mode to prepare it for inference, and load up pretrained weights if available.

In [ ]:
# Define model path and select device
model_path = os.path.join(BASE_PATH, f"model_{N_SAMPLES}.pth")
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model structure
print("Initializing model structure...")
model = GibbsAppellNet().to(device)

# Load pretrained weights if the file exists
if os.path.exists(model_path):
    print(f"Loading weights from: {model_path}")

    # Load state dictionary and map to the appropriate device
    state_dict = torch.load(model_path, map_location=device)

    # Apply the weights to the model
    model.load_state_dict(state_dict)

    # Set the model to evaluation mode for inference
    model.eval()  # Ensures layers like Dropout/BatchNorm behave correctly
    print("Model loaded successfully.")

else:
    print(f"ERROR: Model file not found at {model_path}")


###Closed-loop simulation

First, to avoid using integrator functions that depend on NumPy, we implemented custom ones here.

In [ ]:
def integrate_semi_implicit(q, dq, ddq, dt):
    """
    Advances a second-order dynamical system one time step using the
    semi-implicit (symplectic) Euler integration method.

    This integrator is commonly used for mechanical systems of the form:

        q̈ = f(q, q̇, t)

    where:
        q   = position (generalized coordinates)
        q̇   = velocity
        q̈   = acceleration

    The semi-implicit Euler method updates velocity first using the
    current acceleration, then updates position using the *new* velocity:

        q̇_{n+1} = q̇_n + dt * q̈_n
        q_{n+1} = q_n + dt * q̇_{n+1}

    Compared to explicit Euler, this method has improved numerical
    stability and better energy behavior for many physical systems,
    especially Hamiltonian or mechanical systems.

    Args:
        q: Current position (Tensor or compatible numeric type).
        dq: Current velocity (Tensor or compatible numeric type).
        ddq: Current acceleration (Tensor or compatible numeric type).
        dt: Time step (scalar, float or Tensor).

    Returns:
        q_next: Updated position after one time step.
        dq_next: Updated velocity after one time step.

    """

    # Update velocity
    dq_next = dq + dt * ddq

    # Update position using new velocity
    q_next = q + dt * dq_next

    return q_next, dq_next


In [ ]:
def odeint(func, y0, t, args=()):
    """
    Solves a system of ODEs using the Runge-Kutta 4 (RK4) method.
    Designed to be a drop-in replacement for scipy.integrate.odeint
    but for PyTorch Tensors.

    Args:
        func: Function func(y, t, *args) that returns dy/dt (Tensor).
        y0: Initial state (Tensor).
        t: Time steps (Tensor).
        args: Tuple of additional arguments to pass to func.

    Returns:
        Tensor of shape (len(t), len(y0)) containing the state trajectory.
    """
    # Ensure inputs are tensors on the correct device
    if not torch.is_tensor(y0):
        y0 = torch.tensor(y0)

    # We infer device/dtype from the initial state
    device = y0.device
    dtype = y0.dtype

    if not torch.is_tensor(t):
        t = torch.tensor(t, device=device, dtype=dtype)
    else:
        t = t.to(device=device, dtype=dtype)

    n_steps = len(t)
    solution = torch.empty((n_steps, *y0.shape), device=device, dtype=dtype)
    solution[0] = y0

    curr_y = y0

    # Main Integration Loop
    for i in range(n_steps - 1):
        t_curr = t[i]
        dt = t[i+1] - t_curr

        # k1 = f(y, t)
        k1 = func(curr_y, t_curr, *args)

        # k2 = f(y + 0.5*dt*k1, t + 0.5*dt)
        k2 = func(curr_y + 0.5 * dt * k1, t_curr + 0.5 * dt, *args)

        # k3 = f(y + 0.5*dt*k2, t + 0.5*dt)
        k3 = func(curr_y + 0.5 * dt * k2, t_curr + 0.5 * dt, *args)

        # k4 = f(y + dt*k3, t + dt)
        k4 = func(curr_y + dt * k3, t_curr + dt, *args)

        # Update state
        curr_y = curr_y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        solution[i+1] = curr_y

    return solution


Second, we implement a closed-loop forward dynamics simulation that uses neural optimizer to predict acceleration.


In [ ]:
# GPU-aware wall-clock timing helper (syncs CUDA if needed)
def get_time(start_event=None):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.time()

# If simulation results already exist, load and skip recomputation
if os.path.exists(save_filename):
    print(f"Found existing simulation file: {save_filename}")
    print("Skipping simulations and loading data...")

    data = torch.load(save_filename, map_location='cpu')
    rmse_pos_list = data['rmse_pos']
    rmse_vel_list = data['rmse_vel']
    rmse_acc_list = data['rmse_acc']

    save_needed = False

# Otherwise, run full simulation pipeline from scratch
else:
    print(f"File {save_filename} not found. Starting fresh simulations...")
    save_needed = True

    # Pre-generate random initial conditions for all trajectories
    all_init_states = torch.rand(NUM_TRAJS, 4) - 0.5

    # Initialize metric containers and full trajectory storage
    rmse_pos_list, rmse_vel_list, rmse_acc_list = [], [], []
    all_true_data = {'q': [], 'dq': [], 'ddq': []}
    all_pred_data = {'q': [], 'dq': [], 'ddq': []}

    # Store per-trajectory ground-truth generation time
    t_gen_true_list = []
    # Store per-trajectory neural simulation time
    t_gen_sim_list = []
    # Store per-call neural optimizer execution time
    t_neural_call_list = []

    # Convert physics parameters to Torch tensors once
    m1_t, m2_t = torch.tensor(PARAMS['m1']), torch.tensor(PARAMS['m2'])
    l1_t, l2_t = torch.tensor(PARAMS['l1']), torch.tensor(PARAMS['l2'])
    g_t = torch.tensor(9.81)
    args_torch = (m1_t, m2_t, l1_t, l2_t, g_t)

    # Detect whether CUDA is available for proper timing
    use_cuda = torch.cuda.is_available()
    print(f"Starting Evaluation on {NUM_TRAJS} Trajectories... (Timing GPU: {use_cuda})")

    # Global wall-clock timer for full evaluation
    global_start = time.time()

    # Main loop over trajectories
    for k in range(NUM_TRAJS):
        print(f"Processing trajectory #{k+1} ...")

        # Extract initial state
        init_state = all_init_states[k]
        q_init, dq_init = init_state[:2], init_state[2:]

        # Time full ground-truth trajectory generation
        tic_true = time.time()

        # Compute ground-truth solution via physics ODE
        sol_true = odeint(derive_ode_robust, init_state, T_SPAN, args=args_torch)
        q_true, dq_true = sol_true[:, :2], sol_true[:, 2:]

        # Recompute accelerations explicitly for all timesteps
        ddq_true_list = [derive_ode_robust(sol_true[t], torch.tensor(0.0), *args_torch)[2:]
                         for t in range(len(sol_true))]
        ddq_true = torch.stack(ddq_true_list)

        toc_true = time.time()
        t_gen_true_list.append(toc_true - tic_true)

        # Initialize neural simulation state
        q_curr, dq_curr = q_init.clone(), dq_init.clone()
        prev_acc = torch.zeros(2)
        q_sim_list, dq_sim_list, ddq_sim_list = [q_curr], [dq_curr], []

        # Synchronize and start timing neural rollout
        if use_cuda: torch.cuda.synchronize()
        tic_sim = time.time()

        # Roll out neural dynamics across all timesteps
        for i in range(STEPS):

            # Start timing single neural optimizer call
            if use_cuda:
                start_event = torch.cuda.Event(enable_timing=True)
                end_event = torch.cuda.Event(enable_timing=True)
                start_event.record()
            else:
                tic_opt = time.time()

            # Cold-start stabilization for first step
            if i == 0:
                for _ in range(5):
                    out_raw = neural_optimizer(
                        q_curr, dq_curr, model,
                        X_mean, X_std, S_mean, S_std, chain_rule_scale,
                        prev_ddq_guess=prev_acc
                    )
                    prev_acc = torch.as_tensor(out_raw, dtype=torch.float32)

            # Main neural inference step
            ddq_raw = neural_optimizer(
                q_curr, dq_curr, model,
                X_mean, X_std, S_mean, S_std, chain_rule_scale,
                prev_ddq_guess=prev_acc
            )

            # Stop timing neural call
            if use_cuda:
                end_event.record()
                torch.cuda.synchronize()
                t_neural_call_list.append(start_event.elapsed_time(end_event) / 1000.0)
            else:
                toc_opt = time.time()
                t_neural_call_list.append(toc_opt - tic_opt)

            # Sanitize neural output and update state
            ddq_curr = torch.as_tensor(ddq_raw, dtype=torch.float32)
            ddq_sim_list.append(ddq_curr)
            prev_acc = ddq_curr

            # Integrate one step forward (semi-implicit Euler)
            q_next, dq_next = integrate_semi_implicit(q_curr, dq_curr, ddq_curr, DT_SIM)
            q_curr, dq_curr = q_next, dq_next

            if i < STEPS - 1:
                q_sim_list.append(q_curr)
                dq_sim_list.append(dq_curr)

        # Stop full neural simulation timer
        if use_cuda: torch.cuda.synchronize()
        toc_sim = time.time()
        t_gen_sim_list.append(toc_sim - tic_sim)

        # Stack simulated trajectories into tensors
        q_sim, dq_sim = torch.stack(q_sim_list), torch.stack(dq_sim_list)
        ddq_sim = torch.stack(ddq_sim_list)

        # Compute RMSE across time (dim=0)
        err_pos = torch.sqrt(torch.mean((q_true - q_sim)**2, dim=0))
        err_vel = torch.sqrt(torch.mean((dq_true - dq_sim)**2, dim=0))
        err_acc = torch.sqrt(torch.mean((ddq_true - ddq_sim)**2, dim=0))
        rmse_pos_list.append(err_pos)
        rmse_vel_list.append(err_vel)
        rmse_acc_list.append(err_acc)

        # Store full trajectories for later analysis
        all_true_data['q'].append(q_true)
        all_true_data['dq'].append(dq_true)
        all_true_data['ddq'].append(ddq_true)
        all_pred_data['q'].append(q_sim)
        all_pred_data['dq'].append(dq_sim)
        all_pred_data['ddq'].append(ddq_sim)

    print(f"Evaluation Complete. Total time: {time.time() - global_start:.2f}s")

# Save dataset and timing only if simulations were executed
if save_needed:
    print("Saving full dataset to disk...")
    save_dict = {
        'rmse_pos': torch.stack(rmse_pos_list),
        'rmse_vel': torch.stack(rmse_vel_list),
        'rmse_acc': torch.stack(rmse_acc_list),
        'trial_time': T_SPAN,
        'all_true_q': torch.stack(all_true_data['q']),
        'all_true_dq': torch.stack(all_true_data['dq']),
        'all_true_ddq': torch.stack(all_true_data['ddq']),
        'all_pred_q': torch.stack(all_pred_data['q']),
        'all_pred_dq': torch.stack(all_pred_data['dq']),
        'all_pred_ddq': torch.stack(all_pred_data['ddq'])
    }
    torch.save(save_dict, save_filename)
    print(f"Saved successfully to: {save_filename}")

    # Save detailed timing breakdown
    print("Saving timing statistics to disk...")
    timing_dict = {
        't_gen_true_per_traj': torch.tensor(t_gen_true_list),
        't_gen_sim_per_traj': torch.tensor(t_gen_sim_list),
        't_neural_calls_all': torch.tensor(t_neural_call_list)
    }
    torch.save(timing_dict, timing_filename)
    print(f"Timing stats saved successfully to: {timing_filename}")

else:
    print("Skipped saving (using existing file).")


Third, we demostrate our predictions accurately approximate the numerical solution of the forward dynamics problem.

In [ ]:
# Load previously saved simulation results
filename = save_filename

# Apply Nature-style figure settings globally
single_col, double_col, golden_ratio = set_nature_style()

# Load simulation results from disk
print(f"Loading results from {filename}...")
data = torch.load(filename, map_location='cpu')

# Extract time vector and RMSE metrics
t = data['trial_time']
rmse_pos = data['rmse_pos']
rmse_vel = data['rmse_vel']
rmse_acc = data['rmse_acc']

# Extract full predicted and true trajectories
pred_q   = data['all_pred_q']
true_q   = data['all_true_q']
pred_dq  = data['all_pred_dq']
true_dq  = data['all_true_dq']
pred_ddq = data['all_pred_ddq']
true_ddq = data['all_true_ddq']

# Demo trial index and downsampling for plotting
demo_idx = 0
step_slice = slice(None, None, 10)
COLOR_J1 = '#003049'
COLOR_J2 = '#d62828'

# Function to plot trajectories with spaghetti + demo trial overlay
def plot_trajectory_panel(ax, all_pred, single_true, single_pred, title, ylabel):
    t_plot = t[step_slice].tolist()
    for i in range(len(all_pred)):  # spaghetti
        ax.plot(t_plot, all_pred[i, step_slice, 0].tolist(), color=COLOR_J1, lw=0.1, alpha=0.05)
        ax.plot(t_plot, all_pred[i, step_slice, 1].tolist(), color=COLOR_J2, lw=0.1, alpha=0.05)
    ax.plot(t.tolist(), single_true[:, 0].tolist(), color=COLOR_J1, ls='--', lw=1.5, alpha=0.6)
    ax.plot(t.tolist(), single_true[:, 1].tolist(), color=COLOR_J2, ls='--', lw=1.5, alpha=0.6)
    ax.plot(t.tolist(), single_pred[:, 0].tolist(), color=COLOR_J1, ls='-', lw=0.8)
    ax.plot(t.tolist(), single_pred[:, 1].tolist(), color=COLOR_J2, ls='-', lw=0.8)
    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, float(t.max()))

# Function to plot RMSE boxplots with jittered points
def plot_stats_panel(ax, data_tensor, title, ylabel):
    data_list = [data_tensor[:, 0].tolist(), data_tensor[:, 1].tolist()]
    colors = [COLOR_J1, COLOR_J2]
    bp = ax.boxplot(data_list, positions=[1, 2], widths=0.5,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color='white', linewidth=1.5),
                    boxprops=dict(linewidth=0),
                    whiskerprops=dict(color='gray', linewidth=0.8),
                    capprops=dict(color='gray', linewidth=0.8))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    num_points = data_tensor.shape[0]  # jitter points
    jitter1 = (torch.randn(num_points) * 0.04).tolist()
    jitter2 = (torch.randn(num_points) * 0.04).tolist()
    ax.scatter([1 + j for j in jitter1], data_tensor[:, 0].tolist(), color=COLOR_J1, s=2, alpha=0.3, zorder=3)
    ax.scatter([2 + j for j in jitter2], data_tensor[:, 1].tolist(), color=COLOR_J2, s=2, alpha=0.3, zorder=3)
    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['J1', 'J2'])
    ax.set_ylim(bottom=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Figure dimensions using double column width and golden ratio
fig_width = double_col
fig_height = double_col * golden_ratio
fig, axs = plt.subplots(3, 2,
                        figsize=(fig_width, fig_height),
                        constrained_layout=True,
                        gridspec_kw={'width_ratios': [1.5, 0.8], 'height_ratios': [1, 1, 1]})

# Row 1: Position
plot_trajectory_panel(axs[0,0], pred_q, true_q[demo_idx], pred_q[demo_idx],
                      "a  Position Trajectories", "Angle (rad)")
plot_stats_panel(axs[0,1], rmse_pos, "b  Position RMSE", "RMSE (rad)")
axs[0,0].set_xticklabels([])

# Row 2: Velocity
plot_trajectory_panel(axs[1,0], pred_dq, true_dq[demo_idx], pred_dq[demo_idx],
                      "c  Velocity Trajectories", "Velocity (rad/s)")
plot_stats_panel(axs[1,1], rmse_vel, "d  Velocity RMSE", "RMSE (rad/s)")
axs[1,0].set_xticklabels([])

# Row 3: Acceleration
plot_trajectory_panel(axs[2,0], pred_ddq, true_ddq[demo_idx], pred_ddq[demo_idx],
                      "e  Acceleration Trajectories", "Accel (rad/s²)")
plot_stats_panel(axs[2,1], rmse_acc, "f  Acceleration RMSE", "RMSE (rad/s²)")
axs[2,0].set_xlabel("Time (s)")

# Global legend
legend_elements = [
    Line2D([0], [0], color='gray', lw=1.2, ls='--', alpha=0.6, label='Ground Truth'),
    Line2D([0], [0], color='gray', lw=1.2, ls='-', label='Neural Pred'),
    Line2D([0], [0], color=COLOR_J1, lw=2.0, label='Joint 1'),
    Line2D([0], [0], color=COLOR_J2, lw=2.0, label='Joint 2')
]
axs[0,0].legend(handles=legend_elements, loc='upper right', frameon=False, ncol=2)

# Save and display figure
plt.savefig('FIG_full_kinematics.svg', format='svg', transparent=True)
print(f"Saved to FIG_full_kinematics.svg with size ({fig_width:.2f}, {fig_height:.2f}) inches.")
plt.show()


We then quantify the computational overhead of our optimization pipeline relative to a regularly used forward dynamics solver.

In [ ]:
print(f"Loading statistical data from {timing_filename}...")
data = torch.load(timing_filename, map_location='cpu')

import scipy.stats as scipy_stats

# Convert tensors to numpy arrays for SciPy compatibility
times_true = data['t_gen_true_per_traj'].numpy()
times_sim  = data['t_gen_sim_per_traj'].numpy()

# Basic descriptive statistics
print(f"Data loaded: {len(times_true)} trajectories.")
print(f"Mean Time (Ground Truth): {np.mean(times_true):.4f} s")
print(f"Mean Time (Neural Sim):   {np.mean(times_sim):.4f} s")

# Section header: normality testing
print("\n" + "="*40)
print("1. NORMALITY CHECK (Shapiro-Wilk)")
print("="*40)

# Perform Shapiro-Wilk test (H0: data is normally distributed)
stat_true, p_true = scipy_stats.shapiro(times_true)
stat_sim, p_sim   = scipy_stats.shapiro(times_sim)

# Determine normality at alpha = 0.05
is_normal_true = p_true > 0.05
is_normal_sim  = p_sim > 0.05

# Report normality results
print(f"Ground Truth Dist: p={p_true:.4e} [{'Normal' if is_normal_true else 'Not Normal'}]")
print(f"Neural Sim Dist:   p={p_sim:.4e} [{'Normal' if is_normal_sim else 'Not Normal'}]")

# Section header: hypothesis testing
print("\n" + "="*40)
print("2. SIGNIFICANCE TEST (Sim > True?)")
print("="*40)

# Select parametric or non-parametric test based on normality
if is_normal_true and is_normal_sim:
    print("Using Independent T-Test (Parametric)...")
    test_type = "T-Test"
    stat, p_val = scipy_stats.ttest_ind(times_sim, times_true, alternative='greater')
else:
    print("Using Mann-Whitney U Test (Non-Parametric)...")
    test_type = "Mann-Whitney U"
    stat, p_val = scipy_stats.mannwhitneyu(times_sim, times_true, alternative='greater')

# Report test statistic and p-value
print(f"Test Statistic: {stat:.4f}")
print(f"P-Value:        {p_val:.4e}")

# Interpret result at alpha = 0.05
if p_val < 0.05:
    print(f"\nRESULT: SIGNIFICANT. The Neural Simulation takes significantly more time than Ground Truth generation.")
else:
    print(f"\nRESULT: NOT SIGNIFICANT. No statistical difference found.")

# Store results for downstream plotting/reporting
p_value_result = p_val
test_name_result = test_type


Following the calcualtions of the dealys, we visualize them.

In [ ]:
# Create single-column Nature-style figure for timing comparison
fig, ax = plt.subplots(figsize=(single_col, single_col * golden_ratio),
                       constrained_layout=True)

# Prepare data and labels
data_to_plot = [times_true, times_sim]
labels = ['Ground Truth\n(ODE Solver)', 'Neural Sim\n(Network)']

# Define semantic colors (reference vs new method)
colors = ['#8491B4', '#E64B35']

# 1. Overlay jittered scatter points to show distribution density
# (Drawn first so they sit behind the mean/std markers)
for i, data_arr in enumerate(data_to_plot):
    y = data_arr
    x = np.random.normal(i + 1, 0.04, size=len(y))
    # Slightly lowered alpha so the black error bars stand out more
    ax.scatter(x, y, color=colors[i], alpha=0.4, s=5, zorder=2, edgecolors='none')

# 2. Calculate and plot Mean and Standard Deviation
means = [np.mean(d) for d in data_to_plot]
stds = [np.std(d) for d in data_to_plot]

for i, (mean, std) in enumerate(zip(means, stds)):
    ax.errorbar(i + 1, mean, yerr=std, fmt='o',
                color='black',            # Edge color for the circle
                ecolor='black',           # Color of the whiskers
                elinewidth=1.5,           # Line width of whiskers
                capsize=5,                # Width of the horizontal caps
                capthick=1.5,             # Thickness of the caps
                markersize=7,             # Size of the mean circle
                markerfacecolor='white',  # Clean white center
                markeredgewidth=1.5,
                zorder=4)                 # High z-order to stay on top

# Compute bracket and annotation positions above maximum value
y_max = max(np.max(times_true), np.max(times_sim))
y_bracket_start = y_max * 1.05
y_bracket_mid   = y_max * 1.08
y_text_pos      = y_max * 1.12

# Draw statistical significance bracket
ax.plot([1, 1, 2, 2], [y_bracket_start, y_bracket_mid, y_bracket_mid, y_bracket_start],
        lw=0.5, c='black')

# Convert p-value into standard significance notation
if p_value_result < 0.001:
    sig_str = "***"
elif p_value_result < 0.01:
    sig_str = "**"
elif p_value_result < 0.05:
    sig_str = "*"
else:
    sig_str = "n.s."

# Add p-value and significance stars annotation
ax.text(1.5, y_text_pos,
        f"{sig_str}\n(p = {p_value_result:.1e})",
        ha='center', va='bottom', fontsize=6)

# Apply final scientific formatting and labeling
ax.set_ylabel('Computation Time (s)')
ax.set_title(f'Computational Cost ({test_name_result})', loc='left', fontweight='bold')
ax.set_xticks([1, 2])
ax.set_xticklabels(labels)

# Expand y-axis to accommodate statistical annotation
ax.set_ylim(0, y_text_pos * 1.2)

# Remove unnecessary spines for clean publication aesthetic
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Save final figure
save_path = "FIG_timing_comparison.svg"
plt.savefig(save_path)
print(f"Figure saved to {save_path}")
plt.show()

Finally, we measure the per-step computational cost of the optimization procedure.

In [ ]:
# Load neural timing data and convert to milliseconds
timing_filename = "timing_stats.pt"
data = torch.load(timing_filename, map_location='cpu')
times_ms = data['t_neural_calls_all'].numpy() * 1000.0

# Compute summary statistics for annotation
median_val = np.median(times_ms)
q1 = np.percentile(times_ms, 25)
q3 = np.percentile(times_ms, 75)
iqr = q3 - q1

# Create figure with Nature-style proportions
fig, ax = plt.subplots(figsize=(single_col/1.5, single_col * golden_ratio),
                       constrained_layout=True)

# Draw clean boxplot without outliers
bp = ax.boxplot(times_ms,
                positions=[1],
                widths=0.4,
                patch_artist=True,
                showfliers=False,
                medianprops=dict(color='white', linewidth=1.5),
                boxprops=dict(linewidth=0),
                whiskerprops=dict(color='gray', linewidth=0.8),
                capprops=dict(color='gray', linewidth=0.8))

# Apply custom color styling to the box
bp['boxes'][0].set_facecolor('#4DBBD5')
bp['boxes'][0].set_alpha(0.8)

# Add median and IQR annotation beside the box
stats_text = (f"Median: {median_val:.2f} ms\n"
              f"IQR: {iqr:.2f} ms")

ax.text(1.3, median_val, stats_text,
        fontsize=6, verticalalignment='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, linewidth=0.5))

# Clean up axes for publication styling
ax.set_ylabel('Inference Time (ms)')
ax.set_title('Neural Latency', loc='left', fontweight='bold')
ax.set_xticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

# Save and display figure
save_path = "FIG_neural_latency_boxplot_clean.svg"
plt.savefig(save_path)
print(f"Figure saved to {save_path}")
plt.show()


# Section 6: Hardware description

In the last brief section, we record the system configuration to document for reproducibility.

In [ ]:
def get_hw_info():
    # System RAM
    ram_bytes = psutil.virtual_memory().total
    ram_gb = f"{round(ram_bytes / (1024**3), 1)}GB"

    # CPU Model
    try:
        # Run lscpu command to get the specific model name
        command = "cat /proc/cpuinfo | grep 'model name' | head -1"
        cpu_info = subprocess.check_output(command, shell=True).decode().strip()
        # Clean up string (remove "model name : ")
        cpu_model = cpu_info.split(":")[1].strip()
    except Exception:
        cpu_model = "Unknown CPU"

    # GPU & VRAM & CUDA
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_bytes = torch.cuda.get_device_properties(0).total_memory
        vram_gb = f"{round(vram_bytes / (1024**3), 1)}GB"
        cuda_version = torch.version.cuda
    else:
        gpu_name = "None"
        vram_gb = "0GB"
        cuda_version = "N/A"

    # Software Versions
    python_version = sys.version.split()[0]
    pytorch_version = torch.__version__

    # Generate the Paragraph
    paragraph = (
        f"Network training was conducted on the Google Colab platform. "
        f"The system was equipped with a {gpu_name} GPU ({vram_gb} VRAM), "
        f"a {cpu_model} CPU, and {ram_gb} of system RAM. "
        f"We used Python {python_version}, PyTorch {pytorch_version}, "
        f"and CUDA {cuda_version} for model training."
    )

    return paragraph

# Run and print
print(get_hw_info())

The end!